In [ ]:
%pip install xgboost

In [ ]:
%pip install joblib
# %pip install graphviz

In [ ]:
%pip install seaborn

In [ ]:
%pip install bayesian-optimization

# XGBoost

In [ ]:
# XGBoost tuning, multiple targets, and pooled denormalized metrics (NoIntp+ManVa)

import os
import re
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import sklearn.metrics as sm
import xgboost as xgb
from bayes_opt import BayesianOptimization
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


# Configuration
# names = ["BMP", "COD", "NR", "PR"]
names = ["BMP", "COD", "NR", "PR"]

# The original cell used 13 inputs. Change entries if a target has a different
# number of input columns.
inpt_nums = {
    "BMP": 10,
    "COD": 10,
    "NR": 10,
    "PR": 10,
}

niter = 200
initp = 20

k_out = 5
fo_start = 1


parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"


def make_regressor(params):
    return xgb.XGBRegressor(
        objective="reg:squarederror",
        **params
    )


def relative_error_percent(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    error = actual - np.asarray(predicted, dtype=float)

    return np.divide(
        error * 100.0,
        actual,
        out=np.full_like(error, np.nan),
        where=actual != 0,
    )


def metric_values(actual, predicted):
    mse = sm.mean_squared_error(actual, predicted)

    return {
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(mse, 3),
        "RMSE": round(float(np.sqrt(mse)), 3),
        "R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
    }


def pooled_metrics_row(group, scope, fold_out="All"):
    actual = group["Actual(real)"].to_numpy(dtype=float)
    predicted = group["Prediction(real)"].to_numpy(dtype=float)
    residual = actual - predicted
    relative_error = relative_error_percent(actual, predicted)

    return {
        "Name": group["Name"].iloc[0],
        "Scope": scope,
        "Fold_out": fold_out,
        # "Ref": ref,
        "N_predictions": len(group),
        "Pooled_R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(sm.mean_squared_error(actual, predicted), 3),
        "RE_mean": round(float(np.mean(residual)), 3),
        "RE%_mean": round(float(np.nanmean(relative_error)), 3),
    }


for name in names:
    # tempD = pd.read_excel(
    #     os.path.join(parent_dir, f"Table_S1_v3_pp_num_{name}.xlsx")
    # )
    # study_col = tempD.columns[-1]
    # K = len(tempD[study_col].unique())

    # Preserve the original interpolation-fold selection

    print(f"===== Running {name} =====")
    inpt_num = inpt_nums[name]


    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    path1 = os.path.join(
        parent_dir,
        f"XGBoost {name} Best Models_{timestamp}",
    )
    os.makedirs(path1, exist_ok=True)

    performance_rows = []
    pooled_rows = []

    for fold_no_out in range(fo_start, k_out + 1):
        # Outer held-out test data.
        test_path = os.path.join(
            parent_dir,
            f"Testing_data_ori_{name}-fold{fold_no_out}.csv",
        )
        data2 = pd.read_csv(test_path)


        # Add the remaining original folds to the training data.
        data = pd.read_csv(parent_dir + 'Training_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')
        # extra_frames = []
        # for fold_num in range(1, k_out + 1):
        #     if fold_num==fold_no_out:
        #         continue

        #     extra_path = os.path.join(
        #         parent_dir,
        #         f"Testing_data_ori_{name}-fold{fold_num}.csv",
        #     )
        #     extra_frames.append(pd.read_csv(extra_path))

        # if extra_frames:
        #     data = pd.concat(
        #         extra_frames,
        #         axis=0,
        #         ignore_index=True,
        #     )
        # else:
        #     data = None


        # Randomly split outer-training data into training and validation portions.
        data, data3 = train_test_split(
        data,
        test_size=0.20,
        random_state=42,
        shuffle=True,
        )

        # Fit only on training data; transform all three datasets.
        scaler = StandardScaler()
        scaler.fit(data)

        dataset1 = scaler.transform(data)
        dataset2 = scaler.transform(data2)
        dataset3 = scaler.transform(data3)

        X_training = dataset1[:, :inpt_num]
        Y_training = dataset1[:, inpt_num]

        X_testing = dataset2[:, :inpt_num]
        Y_testing = dataset2[:, inpt_num]

        X_valid = dataset3[:, :inpt_num]
        Y_valid = dataset3[:, inpt_num]

        # Stage 1: tune n_estimators and learning_rate.
        def first_stage_objective(n_estimators, learning_rate):
            model = make_regressor({
                "n_estimators": int(n_estimators),
                "learning_rate": learning_rate,
            })
            model.fit(X_training, Y_training)
            prediction = model.predict(X_valid)
            return sm.r2_score(Y_valid, prediction)

        first_stage = BayesianOptimization(
            f=first_stage_objective,
            pbounds={
                "n_estimators": (100, 1500),
                "learning_rate": (1e-5, 1),
            },
            random_state=1,
        )
        first_stage.maximize(
            n_iter=niter,
            init_points=initp,
        )

        n_estimators = int(
            first_stage.max["params"]["n_estimators"]
        )
        learning_rate = float(
            first_stage.max["params"]["learning_rate"]
        )

        # Stage 2: tune the remaining parameters.
        def second_stage_objective(
            max_depth,
            min_child_weight,
            subsample,
            colsample_bytree,
        ):
            model = make_regressor({
                "n_estimators": n_estimators,
                "learning_rate": learning_rate,
                "max_depth": int(max_depth),
                "min_child_weight": int(min_child_weight),
                "subsample": subsample,
                "colsample_bytree": colsample_bytree,
            })
            model.fit(X_training, Y_training)
            prediction = model.predict(X_valid)
            return sm.r2_score(Y_valid, prediction)

        second_stage = BayesianOptimization(
            f=second_stage_objective,
            pbounds={
                "max_depth": (1, 10),
                "min_child_weight": (1, 10),
                "subsample": (0.5, 1.0),
                "colsample_bytree": (0.5, 1.0),
            },
            random_state=1,
        )
        second_stage.maximize(
            n_iter=niter,
            init_points=initp,
        )
        best_params = second_stage.max["params"]

        final_params = {
            "n_estimators": n_estimators,
            "learning_rate": learning_rate,
            "max_depth": int(best_params["max_depth"]),
            "min_child_weight": int(best_params["min_child_weight"]),
            "subsample": float(best_params["subsample"]),
            "colsample_bytree": float(best_params["colsample_bytree"]),
        }

        model = make_regressor(final_params)
        model.fit(X_training, Y_training)
        prediction = model.predict(X_testing)

        model_name = (
            f"XGB_model_{name}"
            f"_kote_{fold_no_out}"
            f"_ne_{final_params['n_estimators']}"
            f"_lr_{final_params['learning_rate']:.5f}"
            f"_md_{final_params['max_depth']}"
            f"_mcw_{final_params['min_child_weight']}"
            f"_ss_{final_params['subsample']:.3f}"
            f"_cs_{final_params['colsample_bytree']:.3f}"
        )
        
        joblib.dump(
            model,
            os.path.join(path1, model_name + ".joblib"),
        )

        # Convert target values and predictions back to original units.
        ymean = scaler.mean_[inpt_num]
        yscale = scaler.scale_[inpt_num]

        actual_real = Y_testing * yscale + ymean
        predicted_real = prediction * yscale + ymean
        residual_real = actual_real - predicted_real
        residual_percent_real = relative_error_percent(
            actual_real,
            predicted_real,
        )
        # ref = fold_to_study[fold_no_out]

        normalized_metrics = metric_values(Y_testing, prediction)
        real_metrics = metric_values(actual_real, predicted_real)

        performance_rows.append({
            "Name": name,
            "Fold_out": fold_no_out,
            # "Ref": ref,
            # "Fold_in": fold_no_in,
            # "Intp_fold": intp_fold,
            "Model_name": model_name,
            **normalized_metrics,
            "MAE_real": real_metrics["MAE"],
            "MSE_real": real_metrics["MSE"],
            "RMSE_real": real_metrics["RMSE"],
            "R2_real": real_metrics["R2"],
            "RE_mean_real": round(
                float(np.mean(residual_real)),
                3,
            ),
            "RE%_mean_real": round(
                float(np.nanmean(residual_percent_real)),
                3,
            ),
        })

        predictions = pd.DataFrame({
            "Name": name,
            "Fold_out": fold_no_out,
            # "Ref": ref,
            # "Fold_in": fold_no_in,
            # "Intp_fold": intp_fold,
            "Model_name": model_name,
            "Test_row": np.arange(len(Y_testing)),
            "Actual": Y_testing,
            "Prediction": prediction,
            "Actual(real)": actual_real,
            "Prediction(real)": predicted_real,
            "RErr(real)": residual_real,
            "RErr%(real)": residual_percent_real,
        })
        pooled_rows.append(predictions)

        predictions.to_csv(
            os.path.join(
                path1,
                f"Predictions_{name}"
                f"_kote_{fold_no_out}.csv",
                # f"_kova_{fold_no_in}"
                # f"_kitr_{intp_fold}.csv",
            ),
            index=False,
        )

    pd.DataFrame(performance_rows).to_csv(
    os.path.join(path1, "Performance_Summary.csv"),
    index=False,
    )

    pooled_df = pd.concat(pooled_rows, ignore_index=True)
    pooled_df.to_csv(
    os.path.join(path1, "All_Denormalized_Predictions_XGB.csv"),
    index=False,
    )

    pooled_metrics = [pooled_metrics_row(pooled_df, "All")]

    for (fold_out), group in pooled_df.groupby(
    ["Fold_out"]
    ):
        pooled_metrics.append(
            pooled_metrics_row(
                group,
                "Fold_out",
                fold_out=fold_out,
                # ref=ref,
            )
        )

    pd.DataFrame(pooled_metrics).to_csv(
    os.path.join(path1, "Pooled_Denormalized_Metrics_XGB.csv"),
    index=False,
    )

In [ ]:
# XGBoost tuning, multiple targets, and pooled denormalized metrics (with Intp)

import os
import re
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import sklearn.metrics as sm
import xgboost as xgb
from bayes_opt import BayesianOptimization
from sklearn.preprocessing import StandardScaler


# Configuration
names = ["BMP", "COD", "NR", "PR"]

# The original cell used 13 inputs. Change entries if a target has a different
# number of input columns.
inpt_nums = {
    "BMP": 10,
    "COD": 10,
    "NR": 10,
    "PR": 10,
}

niter = 200
initp = 20


parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"


def make_regressor(params):
    return xgb.XGBRegressor(
        objective="reg:squarederror",
        **params
    )


def relative_error_percent(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    error = actual - np.asarray(predicted, dtype=float)

    return np.divide(
        error * 100.0,
        actual,
        out=np.full_like(error, np.nan),
        where=actual != 0,
    )


def metric_values(actual, predicted):
    mse = sm.mean_squared_error(actual, predicted)

    return {
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(mse, 3),
        "RMSE": round(float(np.sqrt(mse)), 3),
        "R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
    }


def pooled_metrics_row(group, scope, fold_out="All", ref="All"):
    actual = group["Actual(real)"].to_numpy(dtype=float)
    predicted = group["Prediction(real)"].to_numpy(dtype=float)
    residual = actual - predicted
    relative_error = relative_error_percent(actual, predicted)

    return {
        "Name": group["Name"].iloc[0],
        "Scope": scope,
        "Fold_out": fold_out,
        "Ref": ref,
        "N_predictions": len(group),
        "Pooled_R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(sm.mean_squared_error(actual, predicted), 3),
        "RE_mean": round(float(np.mean(residual)), 3),
        "RE%_mean": round(float(np.nanmean(relative_error)), 3),
    }


for name in names:
    tempD = pd.read_excel(
        os.path.join(parent_dir, f"Table_S1_v3_pp_num_{name}.xlsx")
    )
    study_col = tempD.columns[-1]
    K = len(tempD[study_col].unique())

    # Preserve the original interpolation-fold selection.
    k_in_start = 1
    k_in = K

    print(f"===== Running {name} =====")
    inpt_num = inpt_nums[name]

    # Map fold number to study number.
    mapping_path = os.path.join(parent_dir, f"Fold_mapping_{name}.csv")
    mapping = pd.read_csv(mapping_path)

    fold_to_study = {
        int(fold): int(re.sub(r"\D", "", str(study)))
        for study, fold in zip(
            mapping["Study_number"],
            mapping["Fold_number"],
        )
    }
    fold_numbers = sorted(fold_to_study)

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    path1 = os.path.join(
        parent_dir,
        f"XGBoost {name} Best Models_{timestamp}",
    )
    os.makedirs(path1, exist_ok=True)

    performance_rows = []
    pooled_rows = []

    for fold_no_out in fold_numbers:
        # Outer held-out test data.
        test_path = os.path.join(
            parent_dir,
            f"Testing_data_ori_{name}-fold{fold_no_out}.csv",
        )
        data2 = pd.read_csv(test_path)

        for fold_no_in in fold_numbers:
            if fold_no_out == fold_no_in:
                continue

            # Validation data for this run.
            validation_path = os.path.join(
                parent_dir,
                f"Testing_data_ori_{name}-fold{fold_no_in}.csv",
            )
            data3 = pd.read_csv(validation_path)

           # Add the remaining original folds to the interpolated training data.
            extra_frames = []
            for fold_num in fold_numbers:
                if fold_num in (fold_no_out, fold_no_in):
                    continue

                extra_path = os.path.join(
                    parent_dir,
                    f"Testing_data_ori_{name}-fold{fold_num}.csv",
                )
                extra_frames.append(pd.read_csv(extra_path))

            if extra_frames:
                data_extra_all = pd.concat(
                    extra_frames,
                    axis=0,
                    ignore_index=True,
                )
            else:
                data_extra_all = None

            for intp_fold in range(k_in_start, k_in + 1):
                training_path = os.path.join(
                    parent_dir,
                    f"Ko={fold_no_out} Ki={intp_fold} {name} intp",
                    f"Training_data_int_{name}-foo{fold_no_out}-foi{intp_fold}.csv",
                )
                data = pd.read_csv(training_path)

                if data_extra_all is not None:
                    data = pd.concat(
                        [data, data_extra_all],
                        axis=0,
                        ignore_index=True,
                    )
                    data = data.drop_duplicates().reset_index(drop=True)

                if not (
                    list(data.columns)
                    == list(data2.columns)
                    == list(data3.columns)
                ):
                    raise ValueError(
                        f"Training, test, and validation columns differ "
                        f"for {name}."
                    )

                if data.shape[1] <= inpt_num:
                    raise ValueError(
                        f"Target column index {inpt_num} is not present "
                        f"in {training_path}."
                    )

                # Fit only on training data; transform all three datasets.
                scaler = StandardScaler()
                scaler.fit(data)

                dataset1 = scaler.transform(data)
                dataset2 = scaler.transform(data2)
                dataset3 = scaler.transform(data3)

                X_training = dataset1[:, :inpt_num]
                Y_training = dataset1[:, inpt_num]

                X_testing = dataset2[:, :inpt_num]
                Y_testing = dataset2[:, inpt_num]

                X_valid = dataset3[:, :inpt_num]
                Y_valid = dataset3[:, inpt_num]

                # Stage 1: tune n_estimators and learning_rate.
                def first_stage_objective(n_estimators, learning_rate):
                    model = make_regressor({
                        "n_estimators": int(n_estimators),
                        "learning_rate": learning_rate,
                    })
                    model.fit(X_training, Y_training)
                    prediction = model.predict(X_valid)
                    return sm.r2_score(Y_valid, prediction)

                first_stage = BayesianOptimization(
                    f=first_stage_objective,
                    pbounds={
                        "n_estimators": (100, 1500),
                        "learning_rate": (1e-5, 1),
                    },
                    random_state=1,
                )
                first_stage.maximize(
                    n_iter=niter,
                    init_points=initp,
                )

                n_estimators = int(
                    first_stage.max["params"]["n_estimators"]
                )
                learning_rate = float(
                    first_stage.max["params"]["learning_rate"]
                )

                # Stage 2: tune the remaining parameters.
                def second_stage_objective(
                    max_depth,
                    min_child_weight,
                    subsample,
                    colsample_bytree,
                ):
                    model = make_regressor({
                        "n_estimators": n_estimators,
                        "learning_rate": learning_rate,
                        "max_depth": int(max_depth),
                        "min_child_weight": int(min_child_weight),
                        "subsample": subsample,
                        "colsample_bytree": colsample_bytree,
                    })
                    model.fit(X_training, Y_training)
                    prediction = model.predict(X_valid)
                    return sm.r2_score(Y_valid, prediction)

                second_stage = BayesianOptimization(
                    f=second_stage_objective,
                    pbounds={
                        "max_depth": (1, 10),
                        "min_child_weight": (1, 10),
                        "subsample": (0.5, 1.0),
                        "colsample_bytree": (0.5, 1.0),
                    },
                    random_state=1,
                )
                second_stage.maximize(
                    n_iter=niter,
                    init_points=initp,
                )
                best_params = second_stage.max["params"]

                model = make_regressor({
                    "n_estimators": n_estimators,
                    "learning_rate": learning_rate,
                    "max_depth": int(best_params["max_depth"]),
                    "min_child_weight": int(
                        best_params["min_child_weight"]
                    ),
                    "subsample": float(best_params["subsample"]),
                    "colsample_bytree": float(
                        best_params["colsample_bytree"]
                    ),
                })
                model.fit(X_training, Y_training)
                prediction = model.predict(X_testing)

                model_name = (
                    f"XGB_model_{name}"
                    f"_kote_{fold_no_out}"
                    f"_kova_{fold_no_in}"
                    f"_kitr_{intp_fold}"
                )
                joblib.dump(
                    model,
                    os.path.join(path1, model_name + ".joblib"),
                )

                # Convert target values and predictions back to original units.
                ymean = scaler.mean_[inpt_num]
                yscale = scaler.scale_[inpt_num]

                actual_real = Y_testing * yscale + ymean
                predicted_real = prediction * yscale + ymean
                residual_real = actual_real - predicted_real
                residual_percent_real = relative_error_percent(
                    actual_real,
                    predicted_real,
                )
                ref = fold_to_study[fold_no_out]

                normalized_metrics = metric_values(Y_testing, prediction)
                real_metrics = metric_values(actual_real, predicted_real)

                performance_rows.append({
                    "Name": name,
                    "Fold_out": fold_no_out,
                    "Ref": ref,
                    "Fold_in": fold_no_in,
                    "Intp_fold": intp_fold,
                    "Model_name": model_name,
                    **normalized_metrics,
                    "MAE_real": real_metrics["MAE"],
                    "MSE_real": real_metrics["MSE"],
                    "RMSE_real": real_metrics["RMSE"],
                    "R2_real": real_metrics["R2"],
                    "RE_mean_real": round(
                        float(np.mean(residual_real)),
                        3,
                    ),
                    "RE%_mean_real": round(
                        float(np.nanmean(residual_percent_real)),
                        3,
                    ),
                })

                predictions = pd.DataFrame({
                    "Name": name,
                    "Fold_out": fold_no_out,
                    "Ref": ref,
                    "Fold_in": fold_no_in,
                    "Intp_fold": intp_fold,
                    "Model_name": model_name,
                    "Test_row": np.arange(len(Y_testing)),
                    "Actual": Y_testing,
                    "Prediction": prediction,
                    "Actual(real)": actual_real,
                    "Prediction(real)": predicted_real,
                    "RErr(real)": residual_real,
                    "RErr%(real)": residual_percent_real,
                })
                pooled_rows.append(predictions)

                predictions.to_csv(
                    os.path.join(
                        path1,
                        f"Predictions_{name}"
                        f"_kote_{fold_no_out}"
                        f"_kova_{fold_no_in}"
                        f"_kitr_{intp_fold}.csv",
                    ),
                    index=False,
                )

    pd.DataFrame(performance_rows).to_csv(
        os.path.join(path1, "Performance_Summary.csv"),
        index=False,
    )

    pooled_df = pd.concat(pooled_rows, ignore_index=True)
    pooled_df.to_csv(
        os.path.join(path1, "All_Denormalized_Predictions.csv"),
        index=False,
    )

    pooled_metrics = [pooled_metrics_row(pooled_df, "All")]

    for (fold_out, ref), group in pooled_df.groupby(
        ["Fold_out", "Ref"]
    ):
        pooled_metrics.append(
            pooled_metrics_row(
                group,
                "Fold_out",
                fold_out=fold_out,
                ref=ref,
            )
        )

    pd.DataFrame(pooled_metrics).to_csv(
        os.path.join(path1, "Pooled_Denormalized_Metrics.csv"),
        index=False,
    )

In [ ]:
#modified for paper rev1
# 1 (small)- XGBoost hyperparameter tuninig using BayesianOptimization in 2 steps
from sklearn.preprocessing import StandardScaler

import os
import numpy as np
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import sklearn.metrics as sm
import pandas as pd

import xgboost as xgb
from bayes_opt import BayesianOptimization
import joblib

from xgboost import plot_tree
from xgboost import plot_importance
# import graphviz

import datetime, os
from datetime import datetime

# Clear any logs from previous runs
# !rm -rf ./logs/
# %load_ext tensorboard

#modeling parameters
model_num=1 #10 #how many model want to save
# iter_num=20 #35 #max iteration number
# tr_fld_num=1 # Training data K-fold number
# epchs=150 #500 #800

niter=200 # how many iter bo will run
initp=20  # how many initial iteration

# stp=0 #0=BM, 1=COD, 2=TN, 3=TP
inpt_num=10 #how mamy input the model has

# names = ["Prd", "OR", "NR", "PR"]
name="PR" #"Prd", "OR", "NR", "PR"
k_out   = 5   # outer folds
k_out1   = 5   # k_out, adjusted outer folds
k_out2   = 5   # k_out, adjusted outer folds
k_in    = 5   # 5, inner folds
fo_start=5 #1,starting fold number for outer loop
fi_start=5 #starting fold number for inner loop
k_in_start=5

stp=3 #0=Prd, 1=OR, 2=NR, 3=PR

# actf='relu' #'relu' 'tanh' 'sigmoid'
# nmin=16
# nmax=128 #256#128
# nstp=8
# lr=1e-2 #1e-2 1e-3 1e-1

#ploting parameters
# unt="" #" (mg/L)"
# data_points=20 #total testing data point (rounded)
# err_lim1=25 #residula % error limit shown in the plot
# err_lim=0.25 #residula error limit shown in the plot
# xmax=3

# Parent Directory path
# parent_dir = "/content/drive/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
# dir_name='my_dir1' # directory name wherer the models will be saved

# Safe timestamp for Windows folder names
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
new_folder = f"XGBoost {name} Best Models_{timestamp}"
# new_folder="Actf="+str(actf)+" Lr="+str(lr)+" "+name+" Best Models" #" 1st Architecture" #" Best Modeles"

# make one shared output folder for all models
path1 = os.path.join(parent_dir, new_folder)
os.makedirs(path1, exist_ok=True)


#output statistical info
norm_prmt = pd.read_csv(parent_dir+'Normalizing_parameters.csv').values
mean9=norm_prmt[inpt_num,stp] #row,col
stdev9=norm_prmt[2*inpt_num+1,stp]

# collect all performance rows here, then save one CSV at the end
performance_rows = []

for fold_no_out in range(fo_start, k_out1 + 1):
    # loading original testing data
    # data2 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')
    # data2 = pd.read_csv(parent_dir + 'K=5 folds '+name+'/Training_data_'+name+'-fold'+str(fold_no_out)+'.csv')
    data2 = pd.read_csv(parent_dir+'Normalized_Data-AMBR_ori_v3_pp_num_'+name+'.csv')

    for fold_no_in in range(fi_start, k_out2 + 1):
        # if fold_no_out == fold_no_in:
        #     continue

        # loading original validation data
        # data3 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_in) + '.csv')
        data3 = pd.read_csv(parent_dir+'Normalized_Data-AMBR_ori_v3_pp_num_'+name+'.csv')

        # # loading original training data from the remaining folds
        # extra_frames = []
        # for num in range(1, k_out + 1):
        #     if num == fold_no_out or num == fold_no_in:
        #         continue
        #     extra_frames.append(
        #         pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(num) + '.csv')
        #     )

        # if len(extra_frames) > 0:
        #     data_extra_all = pd.concat(extra_frames, axis=0, ignore_index=True)
        # else:
        #     data_extra_all = None

        for intp_fold in range(k_in_start, k_in + 1):

            # output statistical info
            # norm_prmt = pd.read_csv(parent_dir+'Normalizing_parameters.csv').values
            # mean9 = norm_prmt[inpt_num, stp]
            # stdev9 = norm_prmt[2*inpt_num+1, stp]

            # loading traninig interpolated data
            # data = pd.read_csv(
            #     parent_dir + 'Ko=' + str(fold_no_out) + ' Ki=' + str(intp_fold) + ' ' + name +
            #     ' intp/Training_data_int_'+name+'-foo' + str(fold_no_out) + '-foi' + str(intp_fold) + '.csv'
            # )
            data = pd.read_csv(parent_dir+'K=5 folds '+name+'/Training_data_'+name+'-fold'+str(intp_fold)+'.csv')

            # # merging the remaining original testing folds outside the loop with training data
            # if data_extra_all is not None:
            #     # Stack one after another (row-wise / vertical append)
            #     data = pd.concat([data,data_extra_all], axis=0, ignore_index=True)
            #     # Optional: remove exact duplicate rows after merge
            #     data = data.drop_duplicates().reset_index(drop=True)
            
            # data_nontesting =pd.concat([data, data3], axis=0, ignore_index=True)
            # scaler = StandardScaler()
            # scaler.fit(data_nontesting)

            # data_norm = scaler.transform(data) #training data normalization
            # dataset1 = data_norm
            dataset1 = data.values
            X_training = dataset1[:, 0:inpt_num]
            Y_training = dataset1[:, inpt_num]

            # data2_norm = scaler.transform(data2) #testing data normalization
            # dataset2 = data2_norm
            dataset2 = data2.values
            X_testing = dataset2[:, 0:inpt_num]
            Y_testing = dataset2[:, inpt_num]

            # data3_norm = scaler.transform(data3) #validation data normalization
            # dataset3 = data3_norm
            dataset3 = data3.values
            X_valid = dataset3[:, 0:inpt_num]
            Y_valid = dataset3[:, inpt_num]

            #1
            def bo_params_xgb(n_estimators,learning_rate):#,max_depth,min_child_weight,subsample,colsample_bytree):

                params = {
                    'n_estimators':int(n_estimators),
                    'learning_rate':learning_rate #,
                    # 'max_depth': int(max_depth),
                    # 'min_child_weight': int(min_child_weight),
                    # 'subsample': subsample,
                    # 'colsample_bytree':colsample_bytree
                }



                xgb_model = xgb.XGBRegressor(
                objective='reg:squarederror',
                # eval_metric='rmse',
                # early_stopping_rounds=25,
                **params #,
                # verbose=False # 1 True False
                )

                xgb_model.fit(
                X_training, Y_training#,
                # eval_set=[(X_valid, Y_valid)]
                )

                Y_pred_all = xgb_model.predict(X_valid)

                scores = sm.r2_score(Y_valid, Y_pred_all) #-sm.mean_squared_error(Y_valid, Y_pred_all) or sm.r2_score(Y_valid, Y_pred_all)
                return scores


            xgb_bo = BayesianOptimization(bo_params_xgb, {'n_estimators':(100,1500),
                                                    'learning_rate':(1e-5, 1)#,
                                                    # 'max_depth': (1, 10),
                                                    # 'min_child_weight': (1, 10),
                                                    # 'subsample':(0.5,1),
                                                    # 'colsample_bytree':(0.5,1)
                                                    }, random_state=1)

            results = xgb_bo.maximize(n_iter=niter, init_points=initp)
            # print(xgb_bo.max)

            n_est=xgb_bo.max['params']['n_estimators']
            l_r=xgb_bo.max['params']['learning_rate']




            #2 and 3 together (better)
            def bo_params_xgb(#n_estimators,learning_rate,
                            max_depth,min_child_weight,
                            subsample,colsample_bytree
                            ):

                params = {
                    'n_estimators':int(n_est),
                    'learning_rate':l_r,
                    'max_depth': int(max_depth),
                    'min_child_weight': int(min_child_weight),
                    'subsample': subsample,
                    'colsample_bytree':colsample_bytree
                }



                xgb_model = xgb.XGBRegressor(
                objective='reg:squarederror',
                # eval_metric='rmse',
                # early_stopping_rounds=25,
                **params #,
                # verbose=False # 1 True False
                )

                xgb_model.fit(
                X_training, Y_training#,
                # eval_set=[(X_valid, Y_valid)]
                )

                Y_pred_all = xgb_model.predict(X_valid)

                scores = sm.r2_score(Y_valid, Y_pred_all) #-sm.mean_squared_error(Y_valid, Y_pred_all) or sm.r2_score(Y_valid, Y_pred_all)
                return scores


            xgb_bo = BayesianOptimization(bo_params_xgb, {
                                                        # 'n_estimators':(100,1500),
                                                        # 'learning_rate':(1e-5, 1),
                                                        'max_depth': (1, 10),
                                                        'min_child_weight': (1, 10),
                                                        'subsample':(0.5,1),
                                                        'colsample_bytree':(0.5,1)
                                                        }, random_state=1)


            results = xgb_bo.maximize(n_iter=niter, init_points=initp)
            # print(xgb_bo.max)


            # Extract the dictionary of best parameters
            best_params_dict = xgb_bo.max['params']

            # Assign each parameter to a separate variable
            # n_estimators = int(best_params_dict['n_estimators'])
            # learning_rate = best_params_dict['learning_rate']
            m_d = best_params_dict['max_depth']
            m_c_w = best_params_dict['min_child_weight']
            s_s = best_params_dict['subsample']
            c_b = best_params_dict['colsample_bytree']



            # XGBoost with custom parameters
            xgb_model = xgb.XGBRegressor(
                objective='reg:squarederror',
                n_estimators=int(n_est),#100,
                learning_rate=l_r,#0.1,
                max_depth=int(m_d),#5,
                min_child_weight=int(m_c_w),
                subsample=s_s,#0.8,
                colsample_bytree=c_b
                # Optional regularization:
                # reg_alpha=0.01 #[0, 0.001, 0.01, 0.1, 1]
                # reg_lambda=2 #[0.5, 1, 2, 5, 10]
            )

            # Train the model
            xgb_model.fit(X_training, Y_training)


            # Make predictions on test data
            Y_pred_all = xgb_model.predict(X_testing)


            # print("Mean absolute error =", round(sm.mean_absolute_error(Y_testing, Y_pred_all), 3))
            # print("Mean squared error =", round(sm.mean_squared_error(Y_testing, Y_pred_all), 3))
            # print("Median absolute error =", round(sm.median_absolute_error(Y_testing, Y_pred_all), 3))
            # print("Variance score =", round(np.var(np.array(Y_testing-Y_pred_all)), 4))
            # print("Explained variance score =", round(sm.explained_variance_score(Y_testing, Y_pred_all), 3))
            # print("R2 score =", round(sm.r2_score(Y_testing, Y_pred_all), 3))

            # n_estimators = int(best_params_dict['n_estimators'])
            # learning_rate = best_params_dict['learning_rate']
            m_d = best_params_dict['max_depth']
            m_c_w = best_params_dict['min_child_weight']
            s_s = best_params_dict['subsample']
            c_b = best_params_dict['colsample_bytree']

            model_name= (
                "XBG_model_" + name +
                "_kote_" + str(fold_no_out) +
                "_kova_" + str(fold_no_in) +
                "_kitr_" + str(intp_fold) +
                # "_trails_" + str(num_trails) +
                "_estm_"+ str(n_est)+
                "_lr_"+ str(l_r)+
                "_mxdep_"+ str(m_d)+
                "mncw"+ str(m_c_w)+
                "ssamp"+ str(s_s)+
                "cols_bt"+ str(c_b)+
                "_mse_" + str(round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)) +
                "_r2_" + str(round(sm.r2_score(Y_testing, Y_pred_all), 3))
            )


            # Define the filename with the parameter information
            # model_filename = parent_dir+model_name+".joblib"
            model_filename = path1+"/"+model_name+".joblib"
            

            # Save the best estimator
            joblib.dump(xgb_model, model_filename)




            # all_trials_performance_results = []

            # train_loss = xgb_model.evaluate(X_training, Y_training, verbose=0)[0]
            # test_loss = xgb_model.evaluate(X_testing, Y_testing, verbose=0)[0]

            # all_trials_performance_results.append({
            #     'keras_tuner_trial_id': trial_id_str,
            #     'training_loss': round(train_loss, 5),
            #     'testing_loss': round(test_loss, 5)
            # })



            # results_df_all_trials = pd.DataFrame(all_trials_performance_results)

            # MAE=[""]
            # MSE=[""]
            # RMSE=[""]
            # R2=[""]
            # RES=[""]
            # RES_P=[""]



            MAE=round(sm.mean_absolute_error(Y_testing, Y_pred_all), 3)
            MSE=round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)
            RMSE=round(sm.root_mean_squared_error(Y_testing, Y_pred_all), 3)
            R2=round(sm.r2_score(Y_testing, Y_pred_all), 3)
            RES=Y_testing - Y_pred_all
            RES_P=(Y_testing - Y_pred_all) * 100 / Y_testing

        

            performance_rows.append({
                'Name': name,
                'Fold_out': fold_no_out,
                'Fold_in': fold_no_in,
                'Intp_fold': intp_fold,
                'Model_name': model_name,
                'MAE': MAE,
                'MSE': MSE,
                'RMSE': RMSE,
                'R2': R2,
                'RES': RES,
                'RES_P': RES_P
            })

            # output_csv_path_all_trials = os.path.join(path1, "all_trials_loss_"+"_kote_"+str(fold_no_out)+"_kova_"+str(fold_no_in)+"_kitr_"+str(intp_fold)+".csv")
            # results_df_all_trials.to_csv(output_csv_path_all_trials, index=False)

            file_name = "Predictions"+"_"+name+"_XGboost"+"_kote_"+str(fold_no_out)+"_kova_"+str(fold_no_in)+"_kitr_"+str(intp_fold)
            my_submission = pd.DataFrame({
                'Actual': Y_testing,
                'Prediction': Y_pred_all,
                'RErr': (Y_testing - Y_pred_all),
                'RErr%': (Y_testing - Y_pred_all) * 100 / Y_testing,
                # 'Actual(real)': Y_testing * scaler.scale_[13] + scaler.mean_[13],
                # 'Prediction(real)': Y_pred_all * scaler.scale_[13] + scaler.mean_[13]
                'Actual(real)': Y_testing*stdev9+mean9,
                'Prediction(real)': Y_pred_all*stdev9+mean9
            })
            # my_submission.to_csv('{}.csv'.format(file_name), index=False)
            my_submission.to_csv(os.path.join(path1,file_name+".csv"), index=False)

            # %cd $path1

# %cd $parent_dir

# save one single summary file for all generated models
performance_summary_df = pd.DataFrame(performance_rows)
performance_summary_df.to_csv(os.path.join(path1, f'Perfornace_Summary.csv'), index=False)

# from google.colab import runtime
# runtime.unassign()

In [ ]:
#Original
# 1 (small)- XGBoost hyperparameter tuninig using BayesianOptimization in 2 steps
from sklearn.preprocessing import StandardScaler

import os
import numpy as np
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import sklearn.metrics as sm
import pandas as pd

import xgboost as xgb
from bayes_opt import BayesianOptimization
import joblib

from xgboost import plot_tree
from xgboost import plot_importance
# import graphviz

import datetime, os
from datetime import datetime

# Clear any logs from previous runs
# !rm -rf ./logs/
# %load_ext tensorboard

#modeling parameters
model_num=1 #10 #how many model want to save
# iter_num=20 #35 #max iteration number
# tr_fld_num=1 # Training data K-fold number
# epchs=150 #500 #800

niter=200 # how many iter bo will run
initp=20  # how many initial iteration

# stp=0 #0=BM, 1=COD, 2=TN, 3=TP
inpt_num=13 #how mamy input the model has

# names = ["BM", "TP", "TN", "COD"]
name="TP" #"BM", "TP", "TN", "COD"
k_out   = 10   # outer folds
k_out1   = 10   # k_out, adjusted outer folds
k_out2   = 10   # k_out, adjusted outer folds
k_in    = 2   # 5, inner folds
fo_start=10 #1,starting fold number for outer loop
fi_start=1 #starting fold number for inner loop
k_in_start=2


# actf='relu' #'relu' 'tanh' 'sigmoid'
# nmin=16
# nmax=128 #256#128
# nstp=8
# lr=1e-2 #1e-2 1e-3 1e-1

#ploting parameters
# unt="" #" (mg/L)"
# data_points=20 #total testing data point (rounded)
# err_lim1=25 #residula % error limit shown in the plot
# err_lim=0.25 #residula error limit shown in the plot
# xmax=3

# Parent Directory path
# parent_dir = "/content/drive/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
# dir_name='my_dir1' # directory name wherer the models will be saved

# Safe timestamp for Windows folder names
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
new_folder = f"XGBoost {name} Best Models_{timestamp}"
# new_folder="Actf="+str(actf)+" Lr="+str(lr)+" "+name+" Best Models" #" 1st Architecture" #" Best Modeles"


# make one shared output folder for all models
path1 = os.path.join(parent_dir, new_folder)
os.makedirs(path1, exist_ok=True)

# collect all performance rows here, then save one CSV at the end
performance_rows = []

for fold_no_out in range(fo_start, k_out1 + 1):
    # loading original testing data
    data2 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')

    for fold_no_in in range(fi_start, k_out2 + 1):
        if fold_no_out == fold_no_in:
            continue

        # loading original validation data
        data3 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_in) + '.csv')

        # loading original training data from the remaining folds
        extra_frames = []
        for num in range(1, k_out + 1):
            if num == fold_no_out or num == fold_no_in:
                continue
            extra_frames.append(
                pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(num) + '.csv')
            )

        if len(extra_frames) > 0:
            data_extra_all = pd.concat(extra_frames, axis=0, ignore_index=True)
        else:
            data_extra_all = None

        for intp_fold in range(k_in_start, k_in + 1):

            # output statistical info
            # norm_prmt = pd.read_csv(parent_dir+'Normalizing_parameters.csv').values
            # mean9 = norm_prmt[inpt_num, stp]
            # stdev9 = norm_prmt[2*inpt_num+1, stp]

            # loading traninig interpolated data
            data = pd.read_csv(
                parent_dir + 'Ko=' + str(fold_no_out) + ' Ki=' + str(intp_fold) + ' ' + name +
                ' intp/Training_data_int_'+name+'-foo' + str(fold_no_out) + '-foi' + str(intp_fold) + '.csv'
            )

            # merging the remaining original testing folds outside the loop with training data
            if data_extra_all is not None:
                # Stack one after another (row-wise / vertical append)
                data = pd.concat([data,data_extra_all], axis=0, ignore_index=True)
                # Optional: remove exact duplicate rows after merge
                data = data.drop_duplicates().reset_index(drop=True)
            
            data_nontesting =pd.concat([data, data3], axis=0, ignore_index=True)
            scaler = StandardScaler()
            scaler.fit(data_nontesting)

            data_norm = scaler.transform(data) #training data normalization
            dataset1 = data_norm
            X_training = dataset1[:, 0:inpt_num]
            Y_training = dataset1[:, inpt_num]

            data2_norm = scaler.transform(data2) #testing data normalization
            dataset2 = data2_norm
            X_testing = dataset2[:, 0:inpt_num]
            Y_testing = dataset2[:, inpt_num]

            data3_norm = scaler.transform(data3) #validation data normalization
            dataset3 = data3_norm
            X_valid = dataset3[:, 0:inpt_num]
            Y_valid = dataset3[:, inpt_num]

            #1
            def bo_params_xgb(n_estimators,learning_rate):#,max_depth,min_child_weight,subsample,colsample_bytree):

                params = {
                    'n_estimators':int(n_estimators),
                    'learning_rate':learning_rate #,
                    # 'max_depth': int(max_depth),
                    # 'min_child_weight': int(min_child_weight),
                    # 'subsample': subsample,
                    # 'colsample_bytree':colsample_bytree
                }



                xgb_model = xgb.XGBRegressor(
                objective='reg:squarederror',
                # eval_metric='rmse',
                # early_stopping_rounds=25,
                **params #,
                # verbose=False # 1 True False
                )

                xgb_model.fit(
                X_training, Y_training#,
                # eval_set=[(X_valid, Y_valid)]
                )

                Y_pred_all = xgb_model.predict(X_valid)

                scores = sm.r2_score(Y_valid, Y_pred_all) #-sm.mean_squared_error(Y_valid, Y_pred_all) or sm.r2_score(Y_valid, Y_pred_all)
                return scores


            xgb_bo = BayesianOptimization(bo_params_xgb, {'n_estimators':(100,1500),
                                                    'learning_rate':(1e-5, 1)#,
                                                    # 'max_depth': (1, 10),
                                                    # 'min_child_weight': (1, 10),
                                                    # 'subsample':(0.5,1),
                                                    # 'colsample_bytree':(0.5,1)
                                                    }, random_state=1)

            results = xgb_bo.maximize(n_iter=niter, init_points=initp)
            # print(xgb_bo.max)

            n_est=xgb_bo.max['params']['n_estimators']
            l_r=xgb_bo.max['params']['learning_rate']




            #2 and 3 together (better)
            def bo_params_xgb(#n_estimators,learning_rate,
                            max_depth,min_child_weight,
                            subsample,colsample_bytree
                            ):

                params = {
                    'n_estimators':int(n_est),
                    'learning_rate':l_r,
                    'max_depth': int(max_depth),
                    'min_child_weight': int(min_child_weight),
                    'subsample': subsample,
                    'colsample_bytree':colsample_bytree
                }



                xgb_model = xgb.XGBRegressor(
                objective='reg:squarederror',
                # eval_metric='rmse',
                # early_stopping_rounds=25,
                **params #,
                # verbose=False # 1 True False
                )

                xgb_model.fit(
                X_training, Y_training#,
                # eval_set=[(X_valid, Y_valid)]
                )

                Y_pred_all = xgb_model.predict(X_valid)

                scores = sm.r2_score(Y_valid, Y_pred_all) #-sm.mean_squared_error(Y_valid, Y_pred_all) or sm.r2_score(Y_valid, Y_pred_all)
                return scores


            xgb_bo = BayesianOptimization(bo_params_xgb, {
                                                        # 'n_estimators':(100,1500),
                                                        # 'learning_rate':(1e-5, 1),
                                                        'max_depth': (1, 10),
                                                        'min_child_weight': (1, 10),
                                                        'subsample':(0.5,1),
                                                        'colsample_bytree':(0.5,1)
                                                        }, random_state=1)


            results = xgb_bo.maximize(n_iter=niter, init_points=initp)
            # print(xgb_bo.max)


            # Extract the dictionary of best parameters
            best_params_dict = xgb_bo.max['params']

            # Assign each parameter to a separate variable
            # n_estimators = int(best_params_dict['n_estimators'])
            # learning_rate = best_params_dict['learning_rate']
            m_d = best_params_dict['max_depth']
            m_c_w = best_params_dict['min_child_weight']
            s_s = best_params_dict['subsample']
            c_b = best_params_dict['colsample_bytree']



            # XGBoost with custom parameters
            xgb_model = xgb.XGBRegressor(
                objective='reg:squarederror',
                n_estimators=int(n_est),#100,
                learning_rate=l_r,#0.1,
                max_depth=int(m_d),#5,
                min_child_weight=int(m_c_w),
                subsample=s_s,#0.8,
                colsample_bytree=c_b
                # Optional regularization:
                # reg_alpha=0.01 #[0, 0.001, 0.01, 0.1, 1]
                # reg_lambda=2 #[0.5, 1, 2, 5, 10]
            )

            # Train the model
            xgb_model.fit(X_training, Y_training)


            # Make predictions on test data
            Y_pred_all = xgb_model.predict(X_testing)


            # print("Mean absolute error =", round(sm.mean_absolute_error(Y_testing, Y_pred_all), 3))
            # print("Mean squared error =", round(sm.mean_squared_error(Y_testing, Y_pred_all), 3))
            # print("Median absolute error =", round(sm.median_absolute_error(Y_testing, Y_pred_all), 3))
            # print("Variance score =", round(np.var(np.array(Y_testing-Y_pred_all)), 4))
            # print("Explained variance score =", round(sm.explained_variance_score(Y_testing, Y_pred_all), 3))
            # print("R2 score =", round(sm.r2_score(Y_testing, Y_pred_all), 3))

            # n_estimators = int(best_params_dict['n_estimators'])
            # learning_rate = best_params_dict['learning_rate']
            m_d = best_params_dict['max_depth']
            m_c_w = best_params_dict['min_child_weight']
            s_s = best_params_dict['subsample']
            c_b = best_params_dict['colsample_bytree']

            model_name= (
                "XBG_model_" + name +
                "_kote_" + str(fold_no_out) +
                "_kova_" + str(fold_no_in) +
                "_kitr_" + str(intp_fold) +
                # "_trails_" + str(num_trails) +
                "_estm_"+ str(n_est)+
                "_lr_"+ str(l_r)+
                "_mxdep_"+ str(m_d)+
                "mncw"+ str(m_c_w)+
                "ssamp"+ str(s_s)+
                "cols_bt"+ str(c_b)+
                "_mse_" + str(round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)) +
                "_r2_" + str(round(sm.r2_score(Y_testing, Y_pred_all), 3))
            )


            # Define the filename with the parameter information
            # model_filename = parent_dir+model_name+".joblib"
            model_filename = path1+"/"+model_name+".joblib"
            

            # Save the best estimator
            joblib.dump(xgb_model, model_filename)




            # all_trials_performance_results = []

            # train_loss = xgb_model.evaluate(X_training, Y_training, verbose=0)[0]
            # test_loss = xgb_model.evaluate(X_testing, Y_testing, verbose=0)[0]

            # all_trials_performance_results.append({
            #     'keras_tuner_trial_id': trial_id_str,
            #     'training_loss': round(train_loss, 5),
            #     'testing_loss': round(test_loss, 5)
            # })



            # results_df_all_trials = pd.DataFrame(all_trials_performance_results)

            # MAE=[""]
            # MSE=[""]
            # RMSE=[""]
            # R2=[""]
            # RES=[""]
            # RES_P=[""]



            MAE=round(sm.mean_absolute_error(Y_testing, Y_pred_all), 3)
            MSE=round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)
            RMSE=round(sm.root_mean_squared_error(Y_testing, Y_pred_all), 3)
            R2=round(sm.r2_score(Y_testing, Y_pred_all), 3)
            RES=Y_testing - Y_pred_all
            RES_P=(Y_testing - Y_pred_all) * 100 / Y_testing

        

            performance_rows.append({
                'Name': name,
                'Fold_out': fold_no_out,
                'Fold_in': fold_no_in,
                'Intp_fold': intp_fold,
                'Model_name': model_name,
                'MAE': MAE,
                'MSE': MSE,
                'RMSE': RMSE,
                'R2': R2,
                'RES': RES,
                'RES_P': RES_P
            })

            # output_csv_path_all_trials = os.path.join(path1, "all_trials_loss_"+"_kote_"+str(fold_no_out)+"_kova_"+str(fold_no_in)+"_kitr_"+str(intp_fold)+".csv")
            # results_df_all_trials.to_csv(output_csv_path_all_trials, index=False)

            file_name = "Predictions"+"_kote_"+str(fold_no_out)+"_kova_"+str(fold_no_in)+"_kitr_"+str(intp_fold)
            my_submission = pd.DataFrame({
                'Actual': Y_testing,
                'Prediction': Y_pred_all,
                'RErr': (Y_testing - Y_pred_all),
                'RErr%': (Y_testing - Y_pred_all) * 100 / Y_testing,
                'Actual(real)': Y_testing * scaler.scale_[13] + scaler.mean_[13],
                'Prediction(real)': Y_pred_all * scaler.scale_[13] + scaler.mean_[13]
            })
            # my_submission.to_csv('{}.csv'.format(file_name), index=False)
            my_submission.to_csv(os.path.join(path1,file_name+".csv"), index=False)

            # %cd $path1

# %cd $parent_dir

# save one single summary file for all generated models
performance_summary_df = pd.DataFrame(performance_rows)
performance_summary_df.to_csv(os.path.join(path1, f'Perfornace_Summary.csv'), index=False)

# from google.colab import runtime
# runtime.unassign()

## No CV for interpolation training

In [ ]:
# 1 (small)- XGBoost hyperparameter tuninig using BayesianOptimization in 2 steps (No CV for interpolation training)
from sklearn.preprocessing import StandardScaler

import os
import numpy as np
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import sklearn.metrics as sm
import pandas as pd

import xgboost as xgb
from bayes_opt import BayesianOptimization
import joblib

from xgboost import plot_tree
from xgboost import plot_importance
# import graphviz

import datetime, os
from datetime import datetime

# Clear any logs from previous runs
# !rm -rf ./logs/
# %load_ext tensorboard

#modeling parameters
model_num=1 #10 #how many model want to save
# iter_num=20 #35 #max iteration number
# tr_fld_num=1 # Training data K-fold number
# epchs=150 #500 #800

niter=200 # how many iter bo will run
initp=20  # how many initial iteration

# stp=0 #0=BM, 1=COD, 2=TN, 3=TP
inpt_num=13 #how mamy input the model has

# names = ["BM", "TP", "TN", "COD"]
name="TP" #"BM", "TP", "TN", "COD"
k_out   = 10   # outer folds
k_out1   = 10   # k_out, adjusted outer folds
k_out2   = 10   # k_out, adjusted outer folds
# k_in    = 2   # 5, inner folds
fo_start=10 #1,starting fold number for outer loop
fi_start=1 #starting fold number for inner loop
# k_in_start=2

Inst="NoIntpCV"

# actf='relu' #'relu' 'tanh' 'sigmoid'
# nmin=16
# nmax=128 #256#128
# nstp=8
# lr=1e-2 #1e-2 1e-3 1e-1

#ploting parameters
# unt="" #" (mg/L)"
# data_points=20 #total testing data point (rounded)
# err_lim1=25 #residula % error limit shown in the plot
# err_lim=0.25 #residula error limit shown in the plot
# xmax=3

# Parent Directory path
# parent_dir = "/content/drive/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
# dir_name='my_dir1' # directory name wherer the models will be saved

# Safe timestamp for Windows folder names
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
new_folder = f"XGBoost {name} Best Models_{timestamp} {Inst}"
# new_folder="Actf="+str(actf)+" Lr="+str(lr)+" "+name+" Best Models" #" 1st Architecture" #" Best Modeles"


# make one shared output folder for all models
path1 = os.path.join(parent_dir, new_folder)
os.makedirs(path1, exist_ok=True)

# collect all performance rows here, then save one CSV at the end
performance_rows = []

for fold_no_out in range(fo_start, k_out1 + 1):
    # loading original testing data
    data2 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')



    for fold_no_in in range(fi_start, k_out2 + 1):
        if fold_no_out == fold_no_in:
            continue

        # loading original validation data
        data3 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_in) + '.csv')

        # loading interpolated traninig data
        data = pd.read_csv(
        parent_dir
        + "Inerpolated_Data_"
        + name
        + "_fold"
        + str(fold_no_out)
        + "_fold"
        + str(fold_no_in)
        + ".csv"
        )

        # loading original training data from the remaining folds
        extra_frames = []
        for num in range(1, k_out + 1):
            if num == fold_no_out or num == fold_no_in:
                continue
            extra_frames.append(
                pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(num) + '.csv')
            )
    
        if len(extra_frames) > 0:
            data_extra_all = pd.concat(extra_frames, axis=0, ignore_index=True)
        else:
            data_extra_all = None
    
            # merging the remaining original testing folds outside the loop with training data
        if data_extra_all is not None:
            # Stack one after another (row-wise / vertical append)
            data = pd.concat([data,data_extra_all], axis=0, ignore_index=True)
            # Optional: remove exact duplicate rows after merge
            data = data.drop_duplicates().reset_index(drop=True)


        # data_nontesting =pd.concat([data, data3], axis=0, ignore_index=True)
        scaler = StandardScaler()
        scaler.fit(data)
    
        data_norm = scaler.transform(data) #training data normalization
        dataset1 = data_norm
        X_training = dataset1[:, 0:inpt_num]
        Y_training = dataset1[:, inpt_num]
    
        data2_norm = scaler.transform(data2) #testing data normalization
        dataset2 = data2_norm
        X_testing = dataset2[:, 0:inpt_num]
        Y_testing = dataset2[:, inpt_num]

        data3_norm = scaler.transform(data3) #validation data normalization
        dataset3 = data3_norm
        X_valid = dataset3[:, 0:inpt_num]
        Y_valid = dataset3[:, inpt_num]


        #1
        def bo_params_xgb(n_estimators,learning_rate):#,max_depth,min_child_weight,subsample,colsample_bytree):

            params = {
                'n_estimators':int(n_estimators),
                'learning_rate':learning_rate #,
                # 'max_depth': int(max_depth),
                # 'min_child_weight': int(min_child_weight),
                # 'subsample': subsample,
                # 'colsample_bytree':colsample_bytree
            }



            xgb_model = xgb.XGBRegressor(
            objective='reg:squarederror',
            # eval_metric='rmse',
            # early_stopping_rounds=25,
            **params #,
            # verbose=False # 1 True False
            )

            xgb_model.fit(
            X_training, Y_training#,
            # eval_set=[(X_valid, Y_valid)]
            )

            Y_pred_all = xgb_model.predict(X_valid)

            scores = sm.r2_score(Y_valid, Y_pred_all) #-sm.mean_squared_error(Y_valid, Y_pred_all) or sm.r2_score(Y_valid, Y_pred_all)
            return scores


        xgb_bo = BayesianOptimization(bo_params_xgb, {'n_estimators':(100,1500),
                                                'learning_rate':(1e-5, 1)#,
                                                # 'max_depth': (1, 10),
                                                # 'min_child_weight': (1, 10),
                                                # 'subsample':(0.5,1),
                                                # 'colsample_bytree':(0.5,1)
                                                }, random_state=1)

        results = xgb_bo.maximize(n_iter=niter, init_points=initp)
        # print(xgb_bo.max)

        n_est=xgb_bo.max['params']['n_estimators']
        l_r=xgb_bo.max['params']['learning_rate']




        #2 and 3 together (better)
        def bo_params_xgb(#n_estimators,learning_rate,
                        max_depth,min_child_weight,
                        subsample,colsample_bytree
                        ):

            params = {
                'n_estimators':int(n_est),
                'learning_rate':l_r,
                'max_depth': int(max_depth),
                'min_child_weight': int(min_child_weight),
                'subsample': subsample,
                'colsample_bytree':colsample_bytree
            }



            xgb_model = xgb.XGBRegressor(
            objective='reg:squarederror',
            # eval_metric='rmse',
            # early_stopping_rounds=25,
            **params #,
            # verbose=False # 1 True False
            )

            xgb_model.fit(
            X_training, Y_training#,
            # eval_set=[(X_valid, Y_valid)]
            )

            Y_pred_all = xgb_model.predict(X_valid)

            scores = sm.r2_score(Y_valid, Y_pred_all) #-sm.mean_squared_error(Y_valid, Y_pred_all) or sm.r2_score(Y_valid, Y_pred_all)
            return scores


        xgb_bo = BayesianOptimization(bo_params_xgb, {
                                                    # 'n_estimators':(100,1500),
                                                    # 'learning_rate':(1e-5, 1),
                                                    'max_depth': (1, 10),
                                                    'min_child_weight': (1, 10),
                                                    'subsample':(0.5,1),
                                                    'colsample_bytree':(0.5,1)
                                                    }, random_state=1)


        results = xgb_bo.maximize(n_iter=niter, init_points=initp)
        # print(xgb_bo.max)


        # Extract the dictionary of best parameters
        best_params_dict = xgb_bo.max['params']

        # Assign each parameter to a separate variable
        # n_estimators = int(best_params_dict['n_estimators'])
        # learning_rate = best_params_dict['learning_rate']
        m_d = best_params_dict['max_depth']
        m_c_w = best_params_dict['min_child_weight']
        s_s = best_params_dict['subsample']
        c_b = best_params_dict['colsample_bytree']



        # XGBoost with custom parameters
        xgb_model = xgb.XGBRegressor(
            objective='reg:squarederror',
            n_estimators=int(n_est),#100,
            learning_rate=l_r,#0.1,
            max_depth=int(m_d),#5,
            min_child_weight=int(m_c_w),
            subsample=s_s,#0.8,
            colsample_bytree=c_b
            # Optional regularization:
            # reg_alpha=0.01 #[0, 0.001, 0.01, 0.1, 1]
            # reg_lambda=2 #[0.5, 1, 2, 5, 10]
        )

        # Train the model
        xgb_model.fit(X_training, Y_training)


        # Make predictions on test data
        Y_pred_all = xgb_model.predict(X_testing)


        # print("Mean absolute error =", round(sm.mean_absolute_error(Y_testing, Y_pred_all), 3))
        # print("Mean squared error =", round(sm.mean_squared_error(Y_testing, Y_pred_all), 3))
        # print("Median absolute error =", round(sm.median_absolute_error(Y_testing, Y_pred_all), 3))
        # print("Variance score =", round(np.var(np.array(Y_testing-Y_pred_all)), 4))
        # print("Explained variance score =", round(sm.explained_variance_score(Y_testing, Y_pred_all), 3))
        # print("R2 score =", round(sm.r2_score(Y_testing, Y_pred_all), 3))

        # n_estimators = int(best_params_dict['n_estimators'])
        # learning_rate = best_params_dict['learning_rate']
        m_d = best_params_dict['max_depth']
        m_c_w = best_params_dict['min_child_weight']
        s_s = best_params_dict['subsample']
        c_b = best_params_dict['colsample_bytree']

        model_name= (
            "XBG_model_" + name +
            "_kote_" + str(fold_no_out) +
            "_kova_" + str(fold_no_in) +
            # "_kitr_" + str(intp_fold) +
            # "_trails_" + str(num_trails) +
            "_estm_"+ str(n_est)+
            "_lr_"+ str(l_r)+
            "_mxdep_"+ str(m_d)+
            "mncw"+ str(m_c_w)+
            "ssamp"+ str(s_s)+
            "cols_bt"+ str(c_b)+
            "_mse_" + str(round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)) +
            "_r2_" + str(round(sm.r2_score(Y_testing, Y_pred_all), 3))
        )


        # Define the filename with the parameter information
        # model_filename = parent_dir+model_name+".joblib"
        model_filename = path1+"/"+model_name+".joblib"
        

        # Save the best estimator
        joblib.dump(xgb_model, model_filename)




        # all_trials_performance_results = []

        # train_loss = xgb_model.evaluate(X_training, Y_training, verbose=0)[0]
        # test_loss = xgb_model.evaluate(X_testing, Y_testing, verbose=0)[0]

        # all_trials_performance_results.append({
        #     'keras_tuner_trial_id': trial_id_str,
        #     'training_loss': round(train_loss, 5),
        #     'testing_loss': round(test_loss, 5)
        # })



        # results_df_all_trials = pd.DataFrame(all_trials_performance_results)

        # MAE=[""]
        # MSE=[""]
        # RMSE=[""]
        # R2=[""]
        # RES=[""]
        # RES_P=[""]



        MAE=round(sm.mean_absolute_error(Y_testing, Y_pred_all), 3)
        MSE=round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)
        RMSE=round(sm.root_mean_squared_error(Y_testing, Y_pred_all), 3)
        R2=round(sm.r2_score(Y_testing, Y_pred_all), 3)
        RES=Y_testing - Y_pred_all
        RES_P=(Y_testing - Y_pred_all) * 100 / Y_testing

    

        performance_rows.append({
            'Name': name,
            'Fold_out': fold_no_out,
            'Fold_in': fold_no_in,
            # 'Intp_fold': intp_fold,
            'Model_name': model_name,
            'MAE': MAE,
            'MSE': MSE,
            'RMSE': RMSE,
            'R2': R2,
            'RES': RES,
            'RES_P': RES_P
        })

        # output_csv_path_all_trials = os.path.join(path1, "all_trials_loss_"+"_kote_"+str(fold_no_out)+"_kova_"+str(fold_no_in)+"_kitr_"+str(intp_fold)+".csv")
        # results_df_all_trials.to_csv(output_csv_path_all_trials, index=False)

        file_name = "Predictions"+"_kote_"+str(fold_no_out)+"_kova_"+str(fold_no_in)
        my_submission = pd.DataFrame({
            'Actual': Y_testing,
            'Prediction': Y_pred_all,
            'RErr': (Y_testing - Y_pred_all),
            'RErr%': (Y_testing - Y_pred_all) * 100 / Y_testing,
            'Actual(real)': Y_testing * scaler.scale_[13] + scaler.mean_[13],
            'Prediction(real)': Y_pred_all * scaler.scale_[13] + scaler.mean_[13]
        })
        # my_submission.to_csv('{}.csv'.format(file_name), index=False)
        my_submission.to_csv(os.path.join(path1,file_name+".csv"), index=False)

        # %cd $path1

# %cd $parent_dir

# save one single summary file for all generated models
performance_summary_df = pd.DataFrame(performance_rows)
performance_summary_df.to_csv(os.path.join(path1, f'Perfornace_Summary.csv'), index=False)

# from google.colab import runtime
# runtime.unassign()

# SVR

In [ ]:
# SVR tuning, multiple targets, and pooled denormalized metrics (NoIntp+ManVa)

import os
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import sklearn.metrics as sm
from bayes_opt import BayesianOptimization
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR


# Configuration
names = ["BMP", "COD", "NR", "PR"]

inpt_nums = {
    "BMP": 10,
    "COD": 10,
    "NR": 10,
    "PR": 10,
}

niter = 200
initp = 20

k_out = 5
fo_start = 1

parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"


def make_regressor(log_c, log_epsilon, log_gamma):
    # hyperparameters are searched on a log10 scale
    return SVR(
        kernel="rbf",
        C=10 ** log_c,
        epsilon=10 ** log_epsilon,
        gamma=10 ** log_gamma,
    )


def relative_error_percent(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    error = actual - np.asarray(predicted, dtype=float)

    return np.divide(
        error * 100.0,
        actual,
        out=np.full_like(error, np.nan),
        where=actual != 0,
    )


def metric_values(actual, predicted):
    mse = sm.mean_squared_error(actual, predicted)

    return {
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(mse, 3),
        "RMSE": round(float(np.sqrt(mse)), 3),
        "R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
    }


def pooled_metrics_row(group, scope, fold_out="All"):
    actual = group["Actual(real)"].to_numpy(dtype=float)
    predicted = group["Prediction(real)"].to_numpy(dtype=float)
    residual = actual - predicted
    relative_error = relative_error_percent(actual, predicted)

    return {
        "Name": group["Name"].iloc[0],
        "Scope": scope,
        "Fold_out": fold_out,
        "N_predictions": len(group),
        "Pooled_R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(sm.mean_squared_error(actual, predicted), 3),
        "RE_mean": round(float(np.mean(residual)), 3),
        "RE%_mean": round(float(np.nanmean(relative_error)), 3),
    }


for name in names:
    print(f"===== Running {name} =====")
    inpt_num = inpt_nums[name]

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    path1 = os.path.join(
        parent_dir,
        f"SVR {name} Best Models_{timestamp}",
    )
    os.makedirs(path1, exist_ok=True)

    performance_rows = []
    pooled_rows = []

    for fold_no_out in range(fo_start, k_out + 1):
        # Outer held-out test data.
        data2 = pd.read_csv(os.path.join(
            parent_dir,
            f"Testing_data_ori_{name}-fold{fold_no_out}.csv",
        ))

        # Remaining original folds form the outer-training data.
        data = pd.read_csv(parent_dir + 'Training_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')
        # extra_frames = []
        # for fold_num in range(1, k_out + 1):
        #     if fold_num == fold_no_out:
        #         continue
        #     extra_frames.append(pd.read_csv(os.path.join(
        #         parent_dir,
        #         f"Testing_data_ori_{name}-fold{fold_num}.csv",
        #     )))
        # data = pd.concat(extra_frames, axis=0, ignore_index=True)

        # Random 20% of outer-training data as validation.
        data, data3 = train_test_split(
            data,
            test_size=0.20,
            random_state=42,
            shuffle=True,
        )

        # Fit only on training data; transform all three datasets.
        scaler = StandardScaler()
        scaler.fit(data)

        dataset1 = scaler.transform(data)
        dataset2 = scaler.transform(data2)
        dataset3 = scaler.transform(data3)

        X_training = dataset1[:, :inpt_num]
        Y_training = dataset1[:, inpt_num]

        X_testing = dataset2[:, :inpt_num]
        Y_testing = dataset2[:, inpt_num]

        X_valid = dataset3[:, :inpt_num]
        Y_valid = dataset3[:, inpt_num]

        # Bayesian optimization of C, epsilon, gamma (validation R2).
        def svr_objective(log_c, log_epsilon, log_gamma):
            model = make_regressor(log_c, log_epsilon, log_gamma)
            model.fit(X_training, Y_training)
            prediction = model.predict(X_valid)
            return sm.r2_score(Y_valid, prediction)

        optimizer = BayesianOptimization(
            f=svr_objective,
            pbounds={
                "log_c": (-2, 3),         # C: 0.01 - 1000
                "log_epsilon": (-3, 0),   # epsilon: 0.001 - 1 (normalized target)
                "log_gamma": (-4, 1),     # gamma: 1e-4 - 10
            },
            random_state=1,
        )
        optimizer.maximize(
            n_iter=niter,
            init_points=initp,
        )
        best_params = optimizer.max["params"]

        model = make_regressor(
            best_params["log_c"],
            best_params["log_epsilon"],
            best_params["log_gamma"],
        )
        model.fit(X_training, Y_training)
        prediction = model.predict(X_testing)

        model_name = (
            f"SVR_model_{name}"
            f"_kote_{fold_no_out}"
            f"_C_{10 ** best_params['log_c']:.4g}"
            f"_eps_{10 ** best_params['log_epsilon']:.4g}"
            f"_gam_{10 ** best_params['log_gamma']:.4g}"
        )

        joblib.dump(
            model,
            os.path.join(path1, model_name + ".joblib"),
        )

        # Convert target values and predictions back to original units.
        ymean = scaler.mean_[inpt_num]
        yscale = scaler.scale_[inpt_num]

        actual_real = Y_testing * yscale + ymean
        predicted_real = prediction * yscale + ymean
        residual_real = actual_real - predicted_real
        residual_percent_real = relative_error_percent(
            actual_real,
            predicted_real,
        )

        normalized_metrics = metric_values(Y_testing, prediction)
        real_metrics = metric_values(actual_real, predicted_real)

        performance_rows.append({
            "Name": name,
            "Fold_out": fold_no_out,
            "Model_name": model_name,
            "C": round(10 ** best_params["log_c"], 5),
            "epsilon": round(10 ** best_params["log_epsilon"], 5),
            "gamma": round(10 ** best_params["log_gamma"], 5),
            "Val_R2": round(optimizer.max["target"], 3),
            **normalized_metrics,
            "MAE_real": real_metrics["MAE"],
            "MSE_real": real_metrics["MSE"],
            "RMSE_real": real_metrics["RMSE"],
            "R2_real": real_metrics["R2"],
            "RE_mean_real": round(float(np.mean(residual_real)), 3),
            "RE%_mean_real": round(float(np.nanmean(residual_percent_real)), 3),
        })

        predictions = pd.DataFrame({
            "Name": name,
            "Fold_out": fold_no_out,
            "Model_name": model_name,
            "Test_row": np.arange(len(Y_testing)),
            "Actual": Y_testing,
            "Prediction": prediction,
            "Actual(real)": actual_real,
            "Prediction(real)": predicted_real,
            "RErr(real)": residual_real,
            "RErr%(real)": residual_percent_real,
        })
        pooled_rows.append(predictions)

        predictions.to_csv(
            os.path.join(path1, f"Predictions_{name}_kote_{fold_no_out}.csv"),
            index=False,
        )

    pd.DataFrame(performance_rows).to_csv(
        os.path.join(path1, "Performance_Summary.csv"),
        index=False,
    )

    pooled_df = pd.concat(pooled_rows, ignore_index=True)
    pooled_df.to_csv(
        os.path.join(path1, "All_Denormalized_Predictions_SVR.csv"),
        index=False,
    )

    pooled_metrics = [pooled_metrics_row(pooled_df, "All")]
    for fold_out, group in pooled_df.groupby("Fold_out"):
        pooled_metrics.append(
            pooled_metrics_row(group, "Fold_out", fold_out=fold_out)
        )

    pd.DataFrame(pooled_metrics).to_csv(
        os.path.join(path1, "Pooled_Denormalized_Metrics_SVR.csv"),
        index=False,
    )

In [ ]:
#1 (small)- SVR hyperparameter tuning using BayesianOptimization in 2 steps
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

import os
import numpy as np
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import sklearn.metrics as sm
import pandas as pd

from bayes_opt import BayesianOptimization
import joblib

import datetime, os
from datetime import datetime

#modeling parameters
model_num=1 #10 #how many model want to save
# iter_num=20 #35 #max iteration number
# tr_fld_num=1 # Training data K-fold number
# epchs=150 #500 #800

niter=200 # how many iter bo will run
initp=20  # how many initial iteration

# stp=0 #0=BM, 1=COD, 2=TN, 3=TP
inpt_num=10 #how mamy input the model has

# names = ["Prd", "OR", "NR", "PR"]
name="PR" #"Prd", "OR", "NR", "PR"
k_out   = 5   # outer folds
k_out1   = 5   # k_out, adjusted outer folds
k_out2   = 5   # k_out, adjusted outer folds
k_in    = 5   # 5, inner folds
fo_start=5 #1,starting fold number for outer loop
fi_start=5 #starting fold number for inner loop
k_in_start=5

stp=3 #0=Prd, 1=OR, 2=NR, 3=PR

# actf='relu' #'relu' 'tanh' 'sigmoid'
# nmin=16
# nmax=128 #256#128
# nstp=8
# lr=1e-2 #1e-2 1e-3 1e-1

#ploting parameters
# unt="" #" (mg/L)"
# data_points=20 #total testing data point (rounded)
# err_lim1=25 #residula % error limit shown in the plot
# err_lim=0.25 #residula error limit shown in the plot
# xmax=3

# Parent Directory path
# parent_dir = "/content/drive/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
# dir_name='my_dir1' # directory name wherer the models will be saved

# Safe timestamp for Windows folder names
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
new_folder = f"SVR {name} Best Models_{timestamp}"
# new_folder="Actf="+str(actf)+" Lr="+str(lr)+" "+name+" Best Models" #" 1st Architecture" #" Best Modeles"

# make one shared output folder for all models
path1 = os.path.join(parent_dir, new_folder)
os.makedirs(path1, exist_ok=True)


#output statistical info
norm_prmt = pd.read_csv(parent_dir+'Normalizing_parameters.csv').values
mean9=norm_prmt[inpt_num,stp] #row,col
stdev9=norm_prmt[2*inpt_num+1,stp]

# collect all performance rows here, then save one CSV at the end
performance_rows = []

for fold_no_out in range(fo_start, k_out1 + 1):
    # loading original testing data
    # data2 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')
    # data2 = pd.read_csv(parent_dir + 'K=5 folds '+name+'/Training_data_'+name+'-fold'+str(fold_no_out)+'.csv')
    data2 = pd.read_csv(parent_dir+'Normalized_Data-AMBR_ori_v3_pp_num_'+name+'.csv')

    for fold_no_in in range(fi_start, k_out2 + 1):
        # if fold_no_out == fold_no_in:
        #     continue

        # loading original validation data
        # data3 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_in) + '.csv')
        data3 = pd.read_csv(parent_dir+'Normalized_Data-AMBR_ori_v3_pp_num_'+name+'.csv')

        # # loading original training data from the remaining folds
        # extra_frames = []
        # for num in range(1, k_out + 1):
        #     if num == fold_no_out or num == fold_no_in:
        #         continue
        #     extra_frames.append(
        #         pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(num) + '.csv')
        #     )

        # if len(extra_frames) > 0:
        #     data_extra_all = pd.concat(extra_frames, axis=0, ignore_index=True)
        # else:
        #     data_extra_all = None

        for intp_fold in range(k_in_start, k_in + 1):

            # output statistical info
            # norm_prmt = pd.read_csv(parent_dir+'Normalizing_parameters.csv').values
            # mean9 = norm_prmt[inpt_num, stp]
            # stdev9 = norm_prmt[2*inpt_num+1, stp]

            # loading traninig interpolated data
            # data = pd.read_csv(
            #     parent_dir + 'Ko=' + str(fold_no_out) + ' Ki=' + str(intp_fold) + ' ' + name +
            #     ' intp/Training_data_int_'+name+'-foo' + str(fold_no_out) + '-foi' + str(intp_fold) + '.csv'
            # )
            data = pd.read_csv(parent_dir+'K=5 folds '+name+'/Training_data_'+name+'-fold'+str(intp_fold)+'.csv')

            # # merging the remaining original testing folds outside the loop with training data
            # if data_extra_all is not None:
            #     # Stack one after another (row-wise / vertical append)
            #     data = pd.concat([data,data_extra_all], axis=0, ignore_index=True)
            #     # Optional: remove exact duplicate rows after merge
            #     data = data.drop_duplicates().reset_index(drop=True)
            
            # data_nontesting =pd.concat([data, data3], axis=0, ignore_index=True)
            # scaler = StandardScaler()
            # scaler.fit(data_nontesting)

            # data_norm = scaler.transform(data) #training data normalization
            # dataset1 = data_norm
            dataset1 = data.values
            X_training = dataset1[:, 0:inpt_num]
            Y_training = dataset1[:, inpt_num]

            # data2_norm = scaler.transform(data2) #testing data normalization
            # dataset2 = data2_norm
            dataset2 = data2.values
            X_testing = dataset2[:, 0:inpt_num]
            Y_testing = dataset2[:, inpt_num]

            # data3_norm = scaler.transform(data3) #validation data normalization
            # dataset3 = data3_norm
            dataset3 = data3.values
            X_valid = dataset3[:, 0:inpt_num]
            Y_valid = dataset3[:, inpt_num]
            

            # Step 1: Optimize C and epsilon (keep gamma fixed as "scale")
            def bo_params_svr(log10_C, log10_epsilon):
                C_val = 10 ** log10_C
                eps_val = 10 ** log10_epsilon

                svr_model = SVR(
                    kernel="rbf",
                    C=C_val,
                    epsilon=eps_val,
                    gamma="scale"
                )

                svr_model.fit(X_training, Y_training)
                Y_pred_valid = svr_model.predict(X_valid)

                scores = sm.r2_score(Y_valid, Y_pred_valid)
                return scores

            svr_bo = BayesianOptimization(
                bo_params_svr,
                {
                    "log10_C": (-2, 3),
                    "log10_epsilon": (-4, 0)
                },
                random_state=1
            )

            results = svr_bo.maximize(n_iter=niter, init_points=initp)

            C_best = 10 ** svr_bo.max["params"]["log10_C"]
            eps_best = 10 ** svr_bo.max["params"]["log10_epsilon"]

            # Step 2: Optimize gamma with best C and epsilon fixed
            def bo_params_svr(log10_gamma):
                gamma_val = 10 ** log10_gamma

                svr_model = SVR(
                    kernel="rbf",
                    C=C_best,
                    epsilon=eps_best,
                    gamma=gamma_val
                )

                svr_model.fit(X_training, Y_training)
                Y_pred_valid = svr_model.predict(X_valid)

                scores = sm.r2_score(Y_valid, Y_pred_valid)
                return scores

            svr_bo = BayesianOptimization(
                bo_params_svr,
                {
                    "log10_gamma": (-4, 1)
                },
                random_state=1
            )

            results = svr_bo.maximize(n_iter=niter, init_points=initp)

            gamma_best = 10 ** svr_bo.max["params"]["log10_gamma"]

            # Final SVR with optimized hyperparameters
            svr_model = SVR(
                kernel="rbf",
                C=C_best,
                epsilon=eps_best,
                gamma=gamma_best
            )

            # Train the model
            svr_model.fit(X_training, Y_training)

            # Make predictions on test data
            Y_pred_all = svr_model.predict(X_testing)

            model_name = (
                "SVR_model_" + name +
                "_kote_" + str(fold_no_out) +
                "_kova_" + str(fold_no_in) +
                "_kitr_" + str(intp_fold) +
                "_C_" + str(round(C_best, 8)) +
                "_eps_" + str(round(eps_best, 8)) +
                "_gma_" + str(round(gamma_best, 8)) +
                "_mse_" + str(round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)) +
                "_r2_" + str(round(sm.r2_score(Y_testing, Y_pred_all), 3))
            )

            model_filename = path1 + "/" + model_name + ".joblib"
            joblib.dump(svr_model, model_filename)

            MAE = round(sm.mean_absolute_error(Y_testing, Y_pred_all), 3)
            MSE = round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)
            RMSE = round(sm.root_mean_squared_error(Y_testing, Y_pred_all), 3)
            R2 = round(sm.r2_score(Y_testing, Y_pred_all), 3)
            RES = Y_testing - Y_pred_all
            RES_P = (Y_testing - Y_pred_all) * 100 / Y_testing

            performance_rows.append({
                "Name": name,
                "Fold_out": fold_no_out,
                "Fold_in": fold_no_in,
                "Intp_fold": intp_fold,
                "Model_name": model_name,
                "MAE": MAE,
                "MSE": MSE,
                "RMSE": RMSE,
                "R2": R2,
                "RES": RES,
                "RES_P": RES_P
            })

            file_name = "Predictions"+"_"+name+"_SVR"+"_kote_"+ str(fold_no_out) + "_kova_" + str(fold_no_in) + "_kitr_" + str(intp_fold)
            my_submission = pd.DataFrame({
                'Actual': Y_testing,
                'Prediction': Y_pred_all,
                'RErr': (Y_testing - Y_pred_all),
                'RErr%': (Y_testing - Y_pred_all) * 100 / Y_testing,
                # 'Actual(real)': Y_testing * scaler.scale_[13] + scaler.mean_[13],
                # 'Prediction(real)': Y_pred_all * scaler.scale_[13] + scaler.mean_[13]
                'Actual(real)': Y_testing*stdev9+mean9,
                'Prediction(real)': Y_pred_all*stdev9+mean9
            })
            my_submission.to_csv(os.path.join(path1, file_name + ".csv"), index=False)

performance_summary_df = pd.DataFrame(performance_rows)
performance_summary_df.to_csv(os.path.join(path1, "Perfornace_Summary.csv"), index=False)


# from google.colab import runtime
# runtime.unassign()

In [ ]:
#1 (small)- SVR hyperparameter tuning using BayesianOptimization (improved: joint BO + CV objective)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import KFold, cross_val_score

import os
import numpy as np
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import sklearn.metrics as sm
import pandas as pd

from bayes_opt import BayesianOptimization
import joblib

import datetime, os
from datetime import datetime

#modeling parameters
model_num = 1 # how many model want to save

niter = 240 # slightly higher for joint BO
initp = 30  # slightly higher for joint BO

inpt_num = 13 # how many input the model has

name ="TP" # "BM", "TP", "TN", "COD"
k_out = 10 # outer folds
k_out1 = 10 # adjusted outer folds
k_out2 = 4 # adjusted outer folds
k_in = 5 # inner folds
fo_start = 10 # starting fold number for outer loop
fi_start = 4 # starting fold number for inner loop
k_in_start = 1

# parent_dir = "/content/drive/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
new_folder = f"SVR {name} Best Models_{timestamp}"

path1 = os.path.join(parent_dir, new_folder)
os.makedirs(path1, exist_ok=True)

performance_rows = []

for fold_no_out in range(fo_start, k_out1 + 1):
    # loading original testing data
    data2 = pd.read_csv(parent_dir + "Testing_data_ori_" + name + "-fold" + str(fold_no_out) + ".csv")

    for fold_no_in in range(fi_start, k_out2 + 1):
        if fold_no_out == fold_no_in:
            continue

        # loading original validation data
        data3 = pd.read_csv(parent_dir + "Testing_data_ori_" + name + "-fold" + str(fold_no_in) + ".csv")

        # loading original training data from the remaining folds
        extra_frames = []
        for num in range(1, k_out + 1):
            if num == fold_no_out or num == fold_no_in:
                continue
            extra_frames.append(
                pd.read_csv(parent_dir + "Testing_data_ori_" + name + "-fold" + str(num) + ".csv")
            )

        if len(extra_frames) > 0:
            data_extra_all = pd.concat(extra_frames, axis=0, ignore_index=True)
        else:
            data_extra_all = None

        for intp_fold in range(k_in_start, k_in + 1):

            # loading training interpolated data
            data = pd.read_csv(
                parent_dir + "Ko=" + str(fold_no_out) + " Ki=" + str(intp_fold) + " " + name +
                " intp/Training_data_int_" + name + "-foo" + str(fold_no_out) + "-foi" + str(intp_fold) + ".csv"
            )

            # merging the remaining original testing folds outside the loop with training data
            if data_extra_all is not None:
                data = pd.concat([data, data_extra_all], axis=0, ignore_index=True)
                data = data.drop_duplicates().reset_index(drop=True)

            # IMPORTANT: fit scaler only on training data (avoid validation leakage)
            scaler = StandardScaler()
            scaler.fit(data)

            data_norm = scaler.transform(data)  # training data normalization
            dataset1 = data_norm
            X_training = dataset1[:, 0:inpt_num]
            Y_training = dataset1[:, inpt_num]

            data2_norm = scaler.transform(data2)  # testing data normalization
            dataset2 = data2_norm
            X_testing = dataset2[:, 0:inpt_num]
            Y_testing = dataset2[:, inpt_num]

            data3_norm = scaler.transform(data3)  # validation data normalization
            dataset3 = data3_norm
            X_valid = dataset3[:, 0:inpt_num]
            Y_valid = dataset3[:, inpt_num]

            # Better SVR tuning: optimize C, epsilon, gamma jointly with CV + validation score
            cv_inner = KFold(n_splits=5, shuffle=True, random_state=1)

            def bo_params_svr(log10_C, log10_epsilon, log10_gamma):
                C_val = 10 ** log10_C
                eps_val = 10 ** log10_epsilon
                gamma_val = 10 ** log10_gamma

                svr_model = SVR(
                    kernel="rbf",
                    C=C_val,
                    epsilon=eps_val,
                    gamma=gamma_val
                )

                # Cross-validated R2 on training split
                cv_scores = cross_val_score(
                    svr_model,
                    X_training,
                    Y_training,
                    cv=cv_inner,
                    scoring="r2",
                    n_jobs=-1
                )
                cv_r2 = float(np.mean(cv_scores))

                # Single validation fold R2 (matches your existing evaluation style)
                svr_model.fit(X_training, Y_training)
                Y_pred_valid = svr_model.predict(X_valid)
                valid_r2 = sm.r2_score(Y_valid, Y_pred_valid)

                # Weighted objective: mostly CV, partly holdout validation
                return 0.8 * cv_r2 + 0.2 * valid_r2

            svr_bo = BayesianOptimization(
                bo_params_svr,
                {
                    "log10_C": (-3, 4),
                    "log10_epsilon": (-5, 0),
                    "log10_gamma": (-6, 1)
                },
                random_state=1
            )

            results = svr_bo.maximize(n_iter=niter, init_points=initp)

            best_params_dict = svr_bo.max["params"]
            C_best = 10 ** best_params_dict["log10_C"]
            eps_best = 10 ** best_params_dict["log10_epsilon"]
            gamma_best = 10 ** best_params_dict["log10_gamma"]

            # Final SVR with optimized hyperparameters
            svr_model = SVR(
                kernel="rbf",
                C=C_best,
                epsilon=eps_best,
                gamma=gamma_best
            )

            # Train the model
            svr_model.fit(X_training, Y_training)

            # Make predictions on test data
            Y_pred_all = svr_model.predict(X_testing)

            model_name = (
                "SVR_model_" + name +
                "_kote_" + str(fold_no_out) +
                "_kova_" + str(fold_no_in) +
                "_kitr_" + str(intp_fold) +
                "_C_" + str(round(C_best, 8)) +
                "_eps_" + str(round(eps_best, 8)) +
                "_gma_" + str(round(gamma_best, 8)) +
                "_mse_" + str(round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)) +
                "_r2_" + str(round(sm.r2_score(Y_testing, Y_pred_all), 3))
            )

            model_filename = path1 + "/" + model_name + ".joblib"
            joblib.dump(svr_model, model_filename)

            MAE = round(sm.mean_absolute_error(Y_testing, Y_pred_all), 3)
            MSE = round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)
            RMSE = round(sm.root_mean_squared_error(Y_testing, Y_pred_all), 3)
            R2 = round(sm.r2_score(Y_testing, Y_pred_all), 3)
            RES = Y_testing - Y_pred_all
            RES_P = (Y_testing - Y_pred_all) * 100 / Y_testing

            performance_rows.append({
                "Name": name,
                "Fold_out": fold_no_out,
                "Fold_in": fold_no_in,
                "Intp_fold": intp_fold,
                "Model_name": model_name,
                "MAE": MAE,
                "MSE": MSE,
                "RMSE": RMSE,
                "R2": R2,
                "RES": RES,
                "RES_P": RES_P
            })

            file_name = "Predictions" + "_kote_" + str(fold_no_out) + "_kova_" + str(fold_no_in) + "_kitr_" + str(intp_fold)
            my_submission = pd.DataFrame({
                "Actual": Y_testing,
                "Prediction": Y_pred_all,
                "RErr": (Y_testing - Y_pred_all),
                "RErr%": (Y_testing - Y_pred_all) * 100 / Y_testing,
                "Actual(real)": Y_testing * scaler.scale_[inpt_num] + scaler.mean_[inpt_num],
                "Prediction(real)": Y_pred_all * scaler.scale_[inpt_num] + scaler.mean_[inpt_num]
            })
            my_submission.to_csv(os.path.join(path1, file_name + ".csv"), index=False)

performance_summary_df = pd.DataFrame(performance_rows)
performance_summary_df.to_csv(os.path.join(path1, "Perfornace_Summary.csv"), index=False)

# from google.colab import runtime
# runtime.unassign()

## No CV for interpolation training

In [ ]:
#1 (small)- SVR hyperparameter tuning using BayesianOptimization in 2 steps
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

import os
import numpy as np
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import sklearn.metrics as sm
import pandas as pd

from bayes_opt import BayesianOptimization
import joblib

import datetime, os
from datetime import datetime

#modeling parameters
model_num = 1 # how many model want to save

niter = 200 # how many iter bo will run
initp = 20 # how many initial iteration

inpt_num = 13 # how many input the model has

name ="TP" # "BM", "TP", "TN", "COD"
k_out = 10 # outer folds
k_out1 = 10 # adjusted outer folds
k_out2 = 10 # adjusted outer folds
fo_start = 10 # starting fold number for outer loop
fi_start = 1 # starting fold number for inner loop

# parent_dir = "/content/drive/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
new_folder = f"SVR {name} Best Models_{timestamp}"

path1 = os.path.join(parent_dir, new_folder)
os.makedirs(path1, exist_ok=True)

performance_rows = []

for fold_no_out in range(fo_start, k_out1 + 1):
    # loading original testing data
    data2 = pd.read_csv(parent_dir + "Testing_data_ori_" + name + "-fold" + str(fold_no_out) + ".csv")

    for fold_no_in in range(fi_start, k_out2 + 1):
        if fold_no_out == fold_no_in:
            continue

        # loading original validation data
        data3 = pd.read_csv(parent_dir + "Testing_data_ori_" + name + "-fold" + str(fold_no_in) + ".csv")

        # loading original training data from the remaining folds
        extra_frames = []
        for num in range(1, k_out + 1):
            if num == fold_no_out or num == fold_no_in:
                continue
            extra_frames.append(
                pd.read_csv(parent_dir + "Testing_data_ori_" + name + "-fold" + str(num) + ".csv")
            )

        if len(extra_frames) > 0:
            data_extra_all = pd.concat(extra_frames, axis=0, ignore_index=True)
        else:
            data_extra_all = None


        # loading training interpolated data
        data = pd.read_csv(
        parent_dir
        + "Inerpolated_Data_"
        + name
        + "_fold"
        + str(fold_no_out)
        + "_fold"
        + str(fold_no_in)
        + ".csv"
        )


        # merging the remaining original testing folds outside the loop with training data
        if data_extra_all is not None:
            data = pd.concat([data, data_extra_all], axis=0, ignore_index=True)
            data = data.drop_duplicates().reset_index(drop=True)

        # data_nontesting = pd.concat([data, data3], axis=0, ignore_index=True)
        scaler = StandardScaler()
        scaler.fit(data)

        data_norm = scaler.transform(data)  # training data normalization
        dataset1 = data_norm
        X_training = dataset1[:, 0:inpt_num]
        Y_training = dataset1[:, inpt_num]

        data2_norm = scaler.transform(data2)  # testing data normalization
        dataset2 = data2_norm
        X_testing = dataset2[:, 0:inpt_num]
        Y_testing = dataset2[:, inpt_num]

        data3_norm = scaler.transform(data3)  # validation data normalization
        dataset3 = data3_norm
        X_valid = dataset3[:, 0:inpt_num]
        Y_valid = dataset3[:, inpt_num]

        # Step 1: Optimize C and epsilon (keep gamma fixed as "scale")
        def bo_params_svr(log10_C, log10_epsilon):
            C_val = 10 ** log10_C
            eps_val = 10 ** log10_epsilon

            svr_model = SVR(
                kernel="rbf",
                C=C_val,
                epsilon=eps_val,
                gamma="scale"
            )

            svr_model.fit(X_training, Y_training)
            Y_pred_valid = svr_model.predict(X_valid)

            scores = sm.r2_score(Y_valid, Y_pred_valid)
            return scores

        svr_bo = BayesianOptimization(
            bo_params_svr,
            {
                "log10_C": (-2, 3),
                "log10_epsilon": (-4, 0)
            },
            random_state=1
        )

        results = svr_bo.maximize(n_iter=niter, init_points=initp)

        C_best = 10 ** svr_bo.max["params"]["log10_C"]
        eps_best = 10 ** svr_bo.max["params"]["log10_epsilon"]

        # Step 2: Optimize gamma with best C and epsilon fixed
        def bo_params_svr(log10_gamma):
            gamma_val = 10 ** log10_gamma

            svr_model = SVR(
                kernel="rbf",
                C=C_best,
                epsilon=eps_best,
                gamma=gamma_val
            )

            svr_model.fit(X_training, Y_training)
            Y_pred_valid = svr_model.predict(X_valid)

            scores = sm.r2_score(Y_valid, Y_pred_valid)
            return scores

        svr_bo = BayesianOptimization(
            bo_params_svr,
            {
                "log10_gamma": (-4, 1)
            },
            random_state=1
        )

        results = svr_bo.maximize(n_iter=niter, init_points=initp)

        gamma_best = 10 ** svr_bo.max["params"]["log10_gamma"]

        # Final SVR with optimized hyperparameters
        svr_model = SVR(
            kernel="rbf",
            C=C_best,
            epsilon=eps_best,
            gamma=gamma_best
        )

        # Train the model
        svr_model.fit(X_training, Y_training)

        # Make predictions on test data
        Y_pred_all = svr_model.predict(X_testing)

        model_name = (
            "SVR_model_" + name +
            "_kote_" + str(fold_no_out) +
            "_kova_" + str(fold_no_in) +
            # "_kitr_" + str(intp_fold) +
            "_C_" + str(round(C_best, 8)) +
            "_eps_" + str(round(eps_best, 8)) +
            "_gma_" + str(round(gamma_best, 8)) +
            "_mse_" + str(round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)) +
            "_r2_" + str(round(sm.r2_score(Y_testing, Y_pred_all), 3))
        )

        model_filename = path1 + "/" + model_name + ".joblib"
        joblib.dump(svr_model, model_filename)

        MAE = round(sm.mean_absolute_error(Y_testing, Y_pred_all), 3)
        MSE = round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)
        RMSE = round(sm.root_mean_squared_error(Y_testing, Y_pred_all), 3)
        R2 = round(sm.r2_score(Y_testing, Y_pred_all), 3)
        RES = Y_testing - Y_pred_all
        RES_P = (Y_testing - Y_pred_all) * 100 / Y_testing

        performance_rows.append({
            "Name": name,
            "Fold_out": fold_no_out,
            "Fold_in": fold_no_in,
            # "Intp_fold": intp_fold,
            "Model_name": model_name,
            "MAE": MAE,
            "MSE": MSE,
            "RMSE": RMSE,
            "R2": R2,
            "RES": RES,
            "RES_P": RES_P
        })

        file_name = "Predictions" + "_kote_" + str(fold_no_out) + "_kova_" + str(fold_no_in)
        my_submission = pd.DataFrame({
            "Actual": Y_testing,
            "Prediction": Y_pred_all,
            "RErr": (Y_testing - Y_pred_all),
            "RErr%": (Y_testing - Y_pred_all) * 100 / Y_testing,
            "Actual(real)": Y_testing * scaler.scale_[inpt_num] + scaler.mean_[inpt_num],
            "Prediction(real)": Y_pred_all * scaler.scale_[inpt_num] + scaler.mean_[inpt_num]
        })
        my_submission.to_csv(os.path.join(path1, file_name + ".csv"), index=False)

performance_summary_df = pd.DataFrame(performance_rows)
performance_summary_df.to_csv(os.path.join(path1, "Perfornace_Summary.csv"), index=False)


# from google.colab import runtime
# runtime.unassign()

# RF

In [ ]:
# Random Forest tuning, multiple targets, and pooled denormalized metrics (NoIntp+ManVa)

import os
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import sklearn.metrics as sm
from bayes_opt import BayesianOptimization
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor


# Configuration
names = ["BMP", "COD", "NR", "PR"]

inpt_nums = {
    "BMP": 10,
    "COD": 10,
    "NR": 10,
    "PR": 10,
}

niter = 100      # 50 RF is slower than SVR; raise if time allows, initial 200
initp = 10      # initial 20

k_out = 5
fo_start = 1

parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"


def make_regressor(n_estimators, max_depth, min_samples_split,
                   min_samples_leaf, max_features):
    return RandomForestRegressor(
        n_estimators=int(n_estimators),
        max_depth=int(max_depth),
        min_samples_split=int(min_samples_split),
        min_samples_leaf=int(min_samples_leaf),
        max_features=float(max_features),
        random_state=42,
        n_jobs=-1,
    )


def relative_error_percent(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    error = actual - np.asarray(predicted, dtype=float)

    return np.divide(
        error * 100.0,
        actual,
        out=np.full_like(error, np.nan),
        where=actual != 0,
    )


def metric_values(actual, predicted):
    mse = sm.mean_squared_error(actual, predicted)

    return {
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(mse, 3),
        "RMSE": round(float(np.sqrt(mse)), 3),
        "R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
    }


def pooled_metrics_row(group, scope, fold_out="All"):
    actual = group["Actual(real)"].to_numpy(dtype=float)
    predicted = group["Prediction(real)"].to_numpy(dtype=float)
    residual = actual - predicted
    relative_error = relative_error_percent(actual, predicted)

    return {
        "Name": group["Name"].iloc[0],
        "Scope": scope,
        "Fold_out": fold_out,
        "N_predictions": len(group),
        "Pooled_R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(sm.mean_squared_error(actual, predicted), 3),
        "RE_mean": round(float(np.mean(residual)), 3),
        "RE%_mean": round(float(np.nanmean(relative_error)), 3),
    }


for name in names:
    print(f"===== Running {name} =====")
    inpt_num = inpt_nums[name]

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    path1 = os.path.join(
        parent_dir,
        f"RF {name} Best Models_{timestamp}",
    )
    os.makedirs(path1, exist_ok=True)

    performance_rows = []
    pooled_rows = []

    for fold_no_out in range(fo_start, k_out + 1):
        # Outer held-out test data.
        data2 = pd.read_csv(os.path.join(
            parent_dir,
            f"Testing_data_ori_{name}-fold{fold_no_out}.csv",
        ))

        # Remaining original folds form the outer-training data.
        data = pd.read_csv(parent_dir + 'Training_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')
        # extra_frames = []
        # for fold_num in range(1, k_out + 1):
        #     if fold_num == fold_no_out:
        #         continue
        #     extra_frames.append(pd.read_csv(os.path.join(
        #         parent_dir,
        #         f"Testing_data_ori_{name}-fold{fold_num}.csv",
        #     )))
        # data = pd.concat(extra_frames, axis=0, ignore_index=True)

        # Random 20% of outer-training data as validation.
        data, data3 = train_test_split(
            data,
            test_size=0.20,
            random_state=42,
            shuffle=True,
        )

        # Fit only on training data; transform all three datasets.
        scaler = StandardScaler()
        scaler.fit(data)

        dataset1 = scaler.transform(data)
        dataset2 = scaler.transform(data2)
        dataset3 = scaler.transform(data3)

        X_training = dataset1[:, :inpt_num]
        Y_training = dataset1[:, inpt_num]

        X_testing = dataset2[:, :inpt_num]
        Y_testing = dataset2[:, inpt_num]

        X_valid = dataset3[:, :inpt_num]
        Y_valid = dataset3[:, inpt_num]

        # Bayesian optimization of RF hyperparameters (validation R2).
        def rf_objective(n_estimators, max_depth, min_samples_split,
                         min_samples_leaf, max_features):
            model = make_regressor(
                n_estimators, max_depth, min_samples_split,
                min_samples_leaf, max_features,
            )
            model.fit(X_training, Y_training)
            prediction = model.predict(X_valid)
            return sm.r2_score(Y_valid, prediction)

        optimizer = BayesianOptimization(
            f=rf_objective,
            pbounds={
                "n_estimators": (100, 1000),
                "max_depth": (2, 30),
                "min_samples_split": (2, 20),
                "min_samples_leaf": (1, 10),
                "max_features": (0.3, 1.0),
            },
            random_state=1,
        )
        optimizer.maximize(
            n_iter=niter,
            init_points=initp,
        )
        best_params = optimizer.max["params"]

        model = make_regressor(**best_params)
        model.fit(X_training, Y_training)
        prediction = model.predict(X_testing)

        model_name = (
            f"RF_model_{name}"
            f"_kote_{fold_no_out}"
            f"_ne_{int(best_params['n_estimators'])}"
            f"_md_{int(best_params['max_depth'])}"
            f"_mss_{int(best_params['min_samples_split'])}"
            f"_msl_{int(best_params['min_samples_leaf'])}"
            f"_mf_{float(best_params['max_features']):.3f}"
        )

        joblib.dump(
            model,
            os.path.join(path1, model_name + ".joblib"),
        )

        # Convert target values and predictions back to original units.
        ymean = scaler.mean_[inpt_num]
        yscale = scaler.scale_[inpt_num]

        actual_real = Y_testing * yscale + ymean
        predicted_real = prediction * yscale + ymean
        residual_real = actual_real - predicted_real
        residual_percent_real = relative_error_percent(
            actual_real,
            predicted_real,
        )

        normalized_metrics = metric_values(Y_testing, prediction)
        real_metrics = metric_values(actual_real, predicted_real)

        performance_rows.append({
            "Name": name,
            "Fold_out": fold_no_out,
            "Model_name": model_name,
            "n_estimators": int(best_params["n_estimators"]),
            "max_depth": int(best_params["max_depth"]),
            "min_samples_split": int(best_params["min_samples_split"]),
            "min_samples_leaf": int(best_params["min_samples_leaf"]),
            "max_features": round(float(best_params["max_features"]), 3),
            "Val_R2": round(optimizer.max["target"], 3),
            **normalized_metrics,
            "MAE_real": real_metrics["MAE"],
            "MSE_real": real_metrics["MSE"],
            "RMSE_real": real_metrics["RMSE"],
            "R2_real": real_metrics["R2"],
            "RE_mean_real": round(float(np.mean(residual_real)), 3),
            "RE%_mean_real": round(float(np.nanmean(residual_percent_real)), 3),
        })

        predictions = pd.DataFrame({
            "Name": name,
            "Fold_out": fold_no_out,
            "Model_name": model_name,
            "Test_row": np.arange(len(Y_testing)),
            "Actual": Y_testing,
            "Prediction": prediction,
            "Actual(real)": actual_real,
            "Prediction(real)": predicted_real,
            "RErr(real)": residual_real,
            "RErr%(real)": residual_percent_real,
        })
        pooled_rows.append(predictions)

        predictions.to_csv(
            os.path.join(path1, f"Predictions_{name}_kote_{fold_no_out}.csv"),
            index=False,
        )

    pd.DataFrame(performance_rows).to_csv(
        os.path.join(path1, "Performance_Summary.csv"),
        index=False,
    )

    pooled_df = pd.concat(pooled_rows, ignore_index=True)
    pooled_df.to_csv(
        os.path.join(path1, "All_Denormalized_Predictions_RF.csv"),
        index=False,
    )

    pooled_metrics = [pooled_metrics_row(pooled_df, "All")]
    for fold_out, group in pooled_df.groupby("Fold_out"):
        pooled_metrics.append(
            pooled_metrics_row(group, "Fold_out", fold_out=fold_out)
        )

    pd.DataFrame(pooled_metrics).to_csv(
        os.path.join(path1, "Pooled_Denormalized_Metrics_RF.csv"),
        index=False,
    )

In [ ]:
# 1 (small)- RandomForest hyperparameter tuninig using BayesianOptimization in 2 steps
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

import os
import numpy as np
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import sklearn.metrics as sm
import pandas as pd

from bayes_opt import BayesianOptimization
import joblib

import datetime, os
from datetime import datetime

# Clear any logs from previous runs
# !rm -rf ./logs/
# %load_ext tensorboard

#modeling parameters
model_num=1 #10 #how many model want to save
# iter_num=20 #35 #max iteration number
# tr_fld_num=1 # Training data K-fold number
# epchs=150 #500 #800

niter=200 # how many iter bo will run
initp=20  # how many initial iteration

# stp=0 #0=BM, 1=COD, 2=TN, 3=TP
inpt_num=10 #how mamy input the model has

# names = ["Prd", "OR", "NR", "PR"]
name="PR" #"Prd", "OR", "NR", "PR"
k_out   = 5   # outer folds
k_out1   = 5   # k_out, adjusted outer folds
k_out2   = 5   # k_out, adjusted outer folds
k_in    = 5   # 5, inner folds
fo_start=5 #1,starting fold number for outer loop
fi_start=5 #starting fold number for inner loop
k_in_start=5

stp=3 #0=Prd, 1=OR, 2=NR, 3=PR

# actf='relu' #'relu' 'tanh' 'sigmoid'
# nmin=16
# nmax=128 #256#128
# nstp=8
# lr=1e-2 #1e-2 1e-3 1e-1

#ploting parameters
# unt="" #" (mg/L)"
# data_points=20 #total testing data point (rounded)
# err_lim1=25 #residula % error limit shown in the plot
# err_lim=0.25 #residula error limit shown in the plot
# xmax=3

# Parent Directory path
# parent_dir = "/content/drive/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
# dir_name='my_dir1' # directory name wherer the models will be saved

# Safe timestamp for Windows folder names
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
new_folder = f"RF {name} Best Models_{timestamp}"
# new_folder="Actf="+str(actf)+" Lr="+str(lr)+" "+name+" Best Models" #" 1st Architecture" #" Best Modeles"

# make one shared output folder for all models
path1 = os.path.join(parent_dir, new_folder)
os.makedirs(path1, exist_ok=True)


#output statistical info
norm_prmt = pd.read_csv(parent_dir+'Normalizing_parameters.csv').values
mean9=norm_prmt[inpt_num,stp] #row,col
stdev9=norm_prmt[2*inpt_num+1,stp]

# collect all performance rows here, then save one CSV at the end
performance_rows = []

for fold_no_out in range(fo_start, k_out1 + 1):
    # loading original testing data
    # data2 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')
    # data2 = pd.read_csv(parent_dir + 'K=5 folds '+name+'/Training_data_'+name+'-fold'+str(fold_no_out)+'.csv')
    data2 = pd.read_csv(parent_dir+'Normalized_Data-AMBR_ori_v3_pp_num_'+name+'.csv')

    for fold_no_in in range(fi_start, k_out2 + 1):
        # if fold_no_out == fold_no_in:
        #     continue

        # loading original validation data
        # data3 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_in) + '.csv')
        data3 = pd.read_csv(parent_dir+'Normalized_Data-AMBR_ori_v3_pp_num_'+name+'.csv')

        # # loading original training data from the remaining folds
        # extra_frames = []
        # for num in range(1, k_out + 1):
        #     if num == fold_no_out or num == fold_no_in:
        #         continue
        #     extra_frames.append(
        #         pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(num) + '.csv')
        #     )

        # if len(extra_frames) > 0:
        #     data_extra_all = pd.concat(extra_frames, axis=0, ignore_index=True)
        # else:
        #     data_extra_all = None

        for intp_fold in range(k_in_start, k_in + 1):

            # output statistical info
            # norm_prmt = pd.read_csv(parent_dir+'Normalizing_parameters.csv').values
            # mean9 = norm_prmt[inpt_num, stp]
            # stdev9 = norm_prmt[2*inpt_num+1, stp]

            # loading traninig interpolated data
            # data = pd.read_csv(
            #     parent_dir + 'Ko=' + str(fold_no_out) + ' Ki=' + str(intp_fold) + ' ' + name +
            #     ' intp/Training_data_int_'+name+'-foo' + str(fold_no_out) + '-foi' + str(intp_fold) + '.csv'
            # )
            data = pd.read_csv(parent_dir+'K=5 folds '+name+'/Training_data_'+name+'-fold'+str(intp_fold)+'.csv')

            # # merging the remaining original testing folds outside the loop with training data
            # if data_extra_all is not None:
            #     # Stack one after another (row-wise / vertical append)
            #     data = pd.concat([data,data_extra_all], axis=0, ignore_index=True)
            #     # Optional: remove exact duplicate rows after merge
            #     data = data.drop_duplicates().reset_index(drop=True)
            
            # data_nontesting =pd.concat([data, data3], axis=0, ignore_index=True)
            # scaler = StandardScaler()
            # scaler.fit(data_nontesting)

            # data_norm = scaler.transform(data) #training data normalization
            # dataset1 = data_norm
            dataset1 = data.values
            X_training = dataset1[:, 0:inpt_num]
            Y_training = dataset1[:, inpt_num]

            # data2_norm = scaler.transform(data2) #testing data normalization
            # dataset2 = data2_norm
            dataset2 = data2.values
            X_testing = dataset2[:, 0:inpt_num]
            Y_testing = dataset2[:, inpt_num]

            # data3_norm = scaler.transform(data3) #validation data normalization
            # dataset3 = data3_norm
            dataset3 = data3.values
            X_valid = dataset3[:, 0:inpt_num]
            Y_valid = dataset3[:, inpt_num]
            

            # 1) Stage-1 BO: n_estimators and max_depth
            def bo_params_rf(n_estimators, max_depth):
                params = {
                    'n_estimators': int(round(n_estimators)),
                    'max_depth': int(round(max_depth)),
                    'random_state': 1,
                    'n_jobs': -1
                }

                rf_model = RandomForestRegressor(**params)
                rf_model.fit(X_training, Y_training)
                Y_pred_valid = rf_model.predict(X_valid)

                scores = sm.r2_score(Y_valid, Y_pred_valid)
                return scores

            rf_bo = BayesianOptimization(
                bo_params_rf,
                {
                    'n_estimators': (100, 1500),
                    'max_depth': (2, 40)
                },
                random_state=1
            )

            results = rf_bo.maximize(n_iter=niter, init_points=initp)

            n_est = int(round(rf_bo.max['params']['n_estimators']))
            m_d = int(round(rf_bo.max['params']['max_depth']))

            # 2) Stage-2 BO: min_samples_split, min_samples_leaf, max_features
            def bo_params_rf(min_samples_split, min_samples_leaf, max_features):
                params = {
                    'n_estimators': n_est,
                    'max_depth': m_d,
                    'min_samples_split': int(round(min_samples_split)),
                    'min_samples_leaf': int(round(min_samples_leaf)),
                    'max_features': max_features,
                    'random_state': 1,
                    'n_jobs': -1
                }

                rf_model = RandomForestRegressor(**params)
                rf_model.fit(X_training, Y_training)
                Y_pred_valid = rf_model.predict(X_valid)

                scores = sm.r2_score(Y_valid, Y_pred_valid)
                return scores

            rf_bo = BayesianOptimization(
                bo_params_rf,
                {
                    'min_samples_split': (2, 20),
                    'min_samples_leaf': (1, 10),
                    'max_features': (0.3, 1.0)
                },
                random_state=1
            )

            results = rf_bo.maximize(n_iter=niter, init_points=initp)

            best_params_dict = rf_bo.max['params']
            m_s_s = int(round(best_params_dict['min_samples_split']))
            m_s_l = int(round(best_params_dict['min_samples_leaf']))
            m_f = best_params_dict['max_features']

            # Final RF model with tuned parameters
            rf_model = RandomForestRegressor(
                n_estimators=n_est,
                max_depth=m_d,
                min_samples_split=m_s_s,
                min_samples_leaf=m_s_l,
                max_features=m_f,
                random_state=1,
                n_jobs=-1
            )

            # Train the model
            rf_model.fit(X_training, Y_training)

            # Make predictions on test data
            Y_pred_all = rf_model.predict(X_testing)

            model_name = (
                "RF_model_" + name +
                "_kote_" + str(fold_no_out) +
                "_kova_" + str(fold_no_in) +
                "_kitr_" + str(intp_fold) +
                "_estm_" + str(n_est) +
                "_mxdep_" + str(m_d) +
                "_mss_" + str(m_s_s) +
                "_msl_" + str(m_s_l) +
                "_mxf_" + str(round(m_f, 4)) +
                "_mse_" + str(round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)) +
                "_r2_" + str(round(sm.r2_score(Y_testing, Y_pred_all), 3))
            )

            # Define the filename with the parameter information
            model_filename = path1 + "/" + model_name + ".joblib"

            # Save the best estimator
            joblib.dump(rf_model, model_filename)

            MAE=round(sm.mean_absolute_error(Y_testing, Y_pred_all), 3)
            MSE=round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)
            RMSE=round(sm.root_mean_squared_error(Y_testing, Y_pred_all), 3)
            R2=round(sm.r2_score(Y_testing, Y_pred_all), 3)
            RES=Y_testing - Y_pred_all
            RES_P=(Y_testing - Y_pred_all) * 100 / Y_testing

            performance_rows.append({
                'Name': name,
                'Fold_out': fold_no_out,
                'Fold_in': fold_no_in,
                'Intp_fold': intp_fold,
                'Model_name': model_name,
                'MAE': MAE,
                'MSE': MSE,
                'RMSE': RMSE,
                'R2': R2,
                'RES': RES,
                'RES_P': RES_P
            })

            file_name = "Predictions"+"_"+name+"_RF"+"_kote_"+str(fold_no_out)+"_kova_"+str(fold_no_in)+"_kitr_"+str(intp_fold)
            my_submission = pd.DataFrame({
                'Actual': Y_testing,
                'Prediction': Y_pred_all,
                'RErr': (Y_testing - Y_pred_all),
                'RErr%': (Y_testing - Y_pred_all) * 100 / Y_testing,
                # 'Actual(real)': Y_testing * scaler.scale_[13] + scaler.mean_[13],
                # 'Prediction(real)': Y_pred_all * scaler.scale_[13] + scaler.mean_[13]
                'Actual(real)': Y_testing*stdev9+mean9,
                'Prediction(real)': Y_pred_all*stdev9+mean9
            })
            my_submission.to_csv(os.path.join(path1,file_name+".csv"), index=False)

# save one single summary file for all generated models
performance_summary_df = pd.DataFrame(performance_rows)
performance_summary_df.to_csv(os.path.join(path1, "Perfornace_Summary.csv"), index=False)

# from google.colab import runtime
# runtime.unassign()

In [ ]:
# 1 (small)- RandomForest hyperparameter tuninig using BayesianOptimization in 2 steps (v2: faster stage-1, richer stage-2)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

import os
import numpy as np
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import sklearn.metrics as sm
import pandas as pd

from bayes_opt import BayesianOptimization
import joblib

import datetime, os
from datetime import datetime

# Clear any logs from previous runs
# !rm -rf ./logs/
# %load_ext tensorboard

#modeling parameters
model_num=1 #10 #how many model want to save
# iter_num=20 #35 #max iteration number
# tr_fld_num=1 # Training data K-fold number
# epchs=150 #500 #800

niter=200 # how many iter bo will run
initp=20  # how many initial iteration

# stp=0 #0=BM, 1=COD, 2=TN, 3=TP
inpt_num=13 #how mamy input the model has

# names = ["BM", "TP", "TN", "COD"]
name="TP" #"BM", "TP", "TN", "COD"
k_out   = 10   # outer folds
k_out1   = 10   # k_out, adjusted outer folds
k_out2   = 10   # k_out, adjusted outer folds
k_in    = 2   # 5, inner folds
fo_start=10 #1,starting fold number for outer loop
fi_start=1 #starting fold number for inner loop
k_in_start=2


# actf='relu' #'relu' 'tanh' 'sigmoid'
# nmin=16
# nmax=128 #256#128
# nstp=8
# lr=1e-2 #1e-2 1e-3 1e-1

#ploting parameters
# unt="" #" (mg/L)"
# data_points=20 #total testing data point (rounded)
# err_lim1=25 #residula % error limit shown in the plot
# err_lim=0.25 #residula error limit shown in the plot
# xmax=3

# Parent Directory path
# parent_dir = "/content/drive/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
# dir_name='my_dir1' # directory name wherer the models will be saved

# Safe timestamp for Windows folder names
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
new_folder = f"RF {name} Best Models_{timestamp}"
# new_folder="Actf="+str(actf)+" Lr="+str(lr)+" "+name+" Best Models" #" 1st Architecture" #" Best Modeles"

# make one shared output folder for all models
path1 = os.path.join(parent_dir, new_folder)
os.makedirs(path1, exist_ok=True)

# collect all performance rows here, then save one CSV at the end
performance_rows = []

for fold_no_out in range(fo_start, k_out1 + 1):
    # loading original testing data
    data2 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')

    for fold_no_in in range(fi_start, k_out2 + 1):
        if fold_no_out == fold_no_in:
            continue

        # loading original validation data
        data3 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_in) + '.csv')

        # loading original training data from the remaining folds
        extra_frames = []
        for num in range(1, k_out + 1):
            if num == fold_no_out or num == fold_no_in:
                continue
            extra_frames.append(
                pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(num) + '.csv')
            )

        if len(extra_frames) > 0:
            data_extra_all = pd.concat(extra_frames, axis=0, ignore_index=True)
        else:
            data_extra_all = None

        for intp_fold in range(k_in_start, k_in + 1):

            # output statistical info
            # norm_prmt = pd.read_csv(parent_dir+'Normalizing_parameters.csv').values
            # mean9 = norm_prmt[inpt_num, stp]
            # stdev9 = norm_prmt[2*inpt_num+1, stp]

            # loading traninig interpolated data
            data = pd.read_csv(
                parent_dir + 'Ko=' + str(fold_no_out) + ' Ki=' + str(intp_fold) + ' ' + name +
                ' intp/Training_data_int_'+name+'-foo' + str(fold_no_out) + '-foi' + str(intp_fold) + '.csv'
            )

            # merging the remaining original testing folds outside the loop with training data
            if data_extra_all is not None:
                # Stack one after another (row-wise / vertical append)
                data = pd.concat([data,data_extra_all], axis=0, ignore_index=True)
                # Optional: remove exact duplicate rows after merge
                data = data.drop_duplicates().reset_index(drop=True)
            
            data_nontesting =pd.concat([data, data3], axis=0, ignore_index=True)
            scaler = StandardScaler()
            scaler.fit(data_nontesting)

            data_norm = scaler.transform(data) #training data normalization
            dataset1 = data_norm
            X_training = dataset1[:, 0:inpt_num]
            Y_training = dataset1[:, inpt_num]

            data2_norm = scaler.transform(data2) #testing data normalization
            dataset2 = data2_norm
            X_testing = dataset2[:, 0:inpt_num]
            Y_testing = dataset2[:, inpt_num]

            data3_norm = scaler.transform(data3) #validation data normalization
            dataset3 = data3_norm
            X_valid = dataset3[:, 0:inpt_num]
            Y_valid = dataset3[:, inpt_num]

            # 1) Stage-1 BO (faster): tune only major structure params
            def bo_params_rf_stage1(n_estimators, max_depth):
                params = {
                    'n_estimators': int(round(n_estimators)),
                    'max_depth': int(round(max_depth)),
                    'random_state': 1,
                    'n_jobs': -1
                }

                rf_model = RandomForestRegressor(**params)
                rf_model.fit(X_training, Y_training)
                Y_pred_valid = rf_model.predict(X_valid)
                return sm.r2_score(Y_valid, Y_pred_valid)

            rf_bo = BayesianOptimization(
                bo_params_rf_stage1,
                {
                    'n_estimators': (100, 1200),
                    'max_depth': (2, 40)
                },
                random_state=1
            )

            results = rf_bo.maximize(n_iter=niter, init_points=initp)

            n_est = int(round(rf_bo.max['params']['n_estimators']))
            m_d = int(round(rf_bo.max['params']['max_depth']))

            # 2) Stage-2 BO (richer): tune split/leaf/max_features + ccp_alpha
            def bo_params_rf_stage2(min_samples_split, min_samples_leaf, max_features, ccp_alpha):
                params = {
                    'n_estimators': n_est,
                    'max_depth': m_d,
                    'min_samples_split': int(round(min_samples_split)),
                    'min_samples_leaf': int(round(min_samples_leaf)),
                    'max_features': max_features,
                    'ccp_alpha': ccp_alpha,
                    'random_state': 1,
                    'n_jobs': -1
                }

                rf_model = RandomForestRegressor(**params)
                rf_model.fit(X_training, Y_training)
                Y_pred_valid = rf_model.predict(X_valid)
                return sm.r2_score(Y_valid, Y_pred_valid)

            rf_bo = BayesianOptimization(
                bo_params_rf_stage2,
                {
                    'min_samples_split': (2, 20),
                    'min_samples_leaf': (1, 10),
                    'max_features': (0.3, 1.0),
                    'ccp_alpha': (0.0, 0.01)
                },
                random_state=1
            )

            results = rf_bo.maximize(n_iter=niter, init_points=initp)

            best_params_dict = rf_bo.max['params']
            m_s_s = int(round(best_params_dict['min_samples_split']))
            m_s_l = int(round(best_params_dict['min_samples_leaf']))
            m_f = best_params_dict['max_features']
            ccp_a = best_params_dict['ccp_alpha']

            # Final RF model with tuned parameters
            rf_model = RandomForestRegressor(
                n_estimators=n_est,
                max_depth=m_d,
                min_samples_split=m_s_s,
                min_samples_leaf=m_s_l,
                max_features=m_f,
                ccp_alpha=ccp_a,
                random_state=1,
                n_jobs=-1
            )

            # Train the model
            rf_model.fit(X_training, Y_training)

            # Make predictions on test data
            Y_pred_all = rf_model.predict(X_testing)

            model_name = (
                "RF_model_" + name +
                "_kote_" + str(fold_no_out) +
                "_kova_" + str(fold_no_in) +
                "_kitr_" + str(intp_fold) +
                "_estm_" + str(n_est) +
                "_mxdep_" + str(m_d) +
                "_mss_" + str(m_s_s) +
                "_msl_" + str(m_s_l) +
                "_mxf_" + str(round(m_f, 4)) +
                "_ccpa_" + str(round(ccp_a, 6)) +
                "_mse_" + str(round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)) +
                "_r2_" + str(round(sm.r2_score(Y_testing, Y_pred_all), 3))
            )

            # Define the filename with the parameter information
            model_filename = path1 + "/" + model_name + ".joblib"

            # Save the best estimator
            joblib.dump(rf_model, model_filename)

            MAE=round(sm.mean_absolute_error(Y_testing, Y_pred_all), 3)
            MSE=round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)
            RMSE=round(sm.root_mean_squared_error(Y_testing, Y_pred_all), 3)
            R2=round(sm.r2_score(Y_testing, Y_pred_all), 3)
            RES=Y_testing - Y_pred_all
            RES_P=(Y_testing - Y_pred_all) * 100 / Y_testing

            performance_rows.append({
                'Name': name,
                'Fold_out': fold_no_out,
                'Fold_in': fold_no_in,
                'Intp_fold': intp_fold,
                'Model_name': model_name,
                'MAE': MAE,
                'MSE': MSE,
                'RMSE': RMSE,
                'R2': R2,
                'RES': RES,
                'RES_P': RES_P
            })

            file_name = "Predictions"+"_kote_"+str(fold_no_out)+"_kova_"+str(fold_no_in)+"_kitr_"+str(intp_fold)
            my_submission = pd.DataFrame({
                'Actual': Y_testing,
                'Prediction': Y_pred_all,
                'RErr': (Y_testing - Y_pred_all),
                'RErr%': (Y_testing - Y_pred_all) * 100 / Y_testing,
                'Actual(real)': Y_testing * scaler.scale_[inpt_num] + scaler.mean_[inpt_num],
                'Prediction(real)': Y_pred_all * scaler.scale_[inpt_num] + scaler.mean_[inpt_num]
            })
            my_submission.to_csv(os.path.join(path1,file_name+".csv"), index=False)

# save one single summary file for all generated models
performance_summary_df = pd.DataFrame(performance_rows)
performance_summary_df.to_csv(os.path.join(path1, "Perfornace_Summary.csv"), index=False)

# from google.colab import runtime
# runtime.unassign()

## No CV for interpolation training

In [ ]:
# 1 (small)- RandomForest hyperparameter tuninig using BayesianOptimization in 2 steps (v2: faster stage-1, richer stage-2)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

import os
import numpy as np
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import sklearn.metrics as sm
import pandas as pd

from bayes_opt import BayesianOptimization
import joblib

import datetime, os
from datetime import datetime

# Clear any logs from previous runs
# !rm -rf ./logs/
# %load_ext tensorboard

#modeling parameters
model_num=1 #10 #how many model want to save
# iter_num=20 #35 #max iteration number
# tr_fld_num=1 # Training data K-fold number
# epchs=150 #500 #800

niter=200 # how many iter bo will run
initp=20  # how many initial iteration

# stp=0 #0=BM, 1=COD, 2=TN, 3=TP
inpt_num=13 #how mamy input the model has

# names = ["BM", "TP", "TN", "COD"]
name="TP" #"BM", "TP", "TN", "COD"
k_out   = 10   # outer folds
k_out1   = 10   # k_out, adjusted outer folds
k_out2   = 10   # k_out, adjusted outer folds
fo_start=10 #1,starting fold number for outer loop
fi_start=1 #starting fold number for inner loop


Inst="NoIntpCV"

# actf='relu' #'relu' 'tanh' 'sigmoid'
# nmin=16
# nmax=128 #256#128
# nstp=8
# lr=1e-2 #1e-2 1e-3 1e-1

#ploting parameters
# unt="" #" (mg/L)"
# data_points=20 #total testing data point (rounded)
# err_lim1=25 #residula % error limit shown in the plot
# err_lim=0.25 #residula error limit shown in the plot
# xmax=3

# Parent Directory path
# parent_dir = "/content/drive/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
# dir_name='my_dir1' # directory name wherer the models will be saved

# Safe timestamp for Windows folder names
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
new_folder = f"RF {name} Best Models_{timestamp} {Inst}"
# new_folder="Actf="+str(actf)+" Lr="+str(lr)+" "+name+" Best Models" #" 1st Architecture" #" Best Modeles"

# make one shared output folder for all models
path1 = os.path.join(parent_dir, new_folder)
os.makedirs(path1, exist_ok=True)

# collect all performance rows here, then save one CSV at the end
performance_rows = []

for fold_no_out in range(fo_start, k_out1 + 1):
    # loading original testing data
    data2 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')

    for fold_no_in in range(fi_start, k_out2 + 1):
        if fold_no_out == fold_no_in:
            continue

        # loading original validation data
        data3 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_in) + '.csv')

        # loading original training data from the remaining folds
        extra_frames = []
        for num in range(1, k_out + 1):
            if num == fold_no_out or num == fold_no_in:
                continue
            extra_frames.append(
                pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(num) + '.csv')
            )

        if len(extra_frames) > 0:
            data_extra_all = pd.concat(extra_frames, axis=0, ignore_index=True)
        else:
            data_extra_all = None


        # loading traninig interpolated data
        data = pd.read_csv(
        parent_dir
        + "Inerpolated_Data_"
        + name
        + "_fold"
        + str(fold_no_out)
        + "_fold"
        + str(fold_no_in)
        + ".csv"
        )


        # merging the remaining original testing folds outside the loop with training data
        if data_extra_all is not None:
            # Stack one after another (row-wise / vertical append)
            data = pd.concat([data,data_extra_all], axis=0, ignore_index=True)
            # Optional: remove exact duplicate rows after merge
            data = data.drop_duplicates().reset_index(drop=True)
        
        # data_nontesting =pd.concat([data, data3], axis=0, ignore_index=True)
        scaler = StandardScaler()
        scaler.fit(data)

        data_norm = scaler.transform(data) #training data normalization
        dataset1 = data_norm
        X_training = dataset1[:, 0:inpt_num]
        Y_training = dataset1[:, inpt_num]

        data2_norm = scaler.transform(data2) #testing data normalization
        dataset2 = data2_norm
        X_testing = dataset2[:, 0:inpt_num]
        Y_testing = dataset2[:, inpt_num]

        data3_norm = scaler.transform(data3) #validation data normalization
        dataset3 = data3_norm
        X_valid = dataset3[:, 0:inpt_num]
        Y_valid = dataset3[:, inpt_num]

        # 1) Stage-1 BO (faster): tune only major structure params
        def bo_params_rf_stage1(n_estimators, max_depth):
            params = {
                'n_estimators': int(round(n_estimators)),
                'max_depth': int(round(max_depth)),
                'random_state': 1,
                'n_jobs': -1
            }

            rf_model = RandomForestRegressor(**params)
            rf_model.fit(X_training, Y_training)
            Y_pred_valid = rf_model.predict(X_valid)
            return sm.r2_score(Y_valid, Y_pred_valid)

        rf_bo = BayesianOptimization(
            bo_params_rf_stage1,
            {
                'n_estimators': (100, 1200),
                'max_depth': (2, 40)
            },
            random_state=1
        )

        results = rf_bo.maximize(n_iter=niter, init_points=initp)

        n_est = int(round(rf_bo.max['params']['n_estimators']))
        m_d = int(round(rf_bo.max['params']['max_depth']))

        # 2) Stage-2 BO (richer): tune split/leaf/max_features + ccp_alpha
        def bo_params_rf_stage2(min_samples_split, min_samples_leaf, max_features, ccp_alpha):
            params = {
                'n_estimators': n_est,
                'max_depth': m_d,
                'min_samples_split': int(round(min_samples_split)),
                'min_samples_leaf': int(round(min_samples_leaf)),
                'max_features': max_features,
                'ccp_alpha': ccp_alpha,
                'random_state': 1,
                'n_jobs': -1
            }

            rf_model = RandomForestRegressor(**params)
            rf_model.fit(X_training, Y_training)
            Y_pred_valid = rf_model.predict(X_valid)
            return sm.r2_score(Y_valid, Y_pred_valid)

        rf_bo = BayesianOptimization(
            bo_params_rf_stage2,
            {
                'min_samples_split': (2, 20),
                'min_samples_leaf': (1, 10),
                'max_features': (0.3, 1.0),
                'ccp_alpha': (0.0, 0.01)
            },
            random_state=1
        )

        results = rf_bo.maximize(n_iter=niter, init_points=initp)

        best_params_dict = rf_bo.max['params']
        m_s_s = int(round(best_params_dict['min_samples_split']))
        m_s_l = int(round(best_params_dict['min_samples_leaf']))
        m_f = best_params_dict['max_features']
        ccp_a = best_params_dict['ccp_alpha']

        # Final RF model with tuned parameters
        rf_model = RandomForestRegressor(
            n_estimators=n_est,
            max_depth=m_d,
            min_samples_split=m_s_s,
            min_samples_leaf=m_s_l,
            max_features=m_f,
            ccp_alpha=ccp_a,
            random_state=1,
            n_jobs=-1
        )

        # Train the model
        rf_model.fit(X_training, Y_training)

        # Make predictions on test data
        Y_pred_all = rf_model.predict(X_testing)

        model_name = (
            "RF_model_" + name +
            "_kote_" + str(fold_no_out) +
            "_kova_" + str(fold_no_in) +
            # "_kitr_" + str(intp_fold) +
            "_estm_" + str(n_est) +
            "_mxdep_" + str(m_d) +
            "_mss_" + str(m_s_s) +
            "_msl_" + str(m_s_l) +
            "_mxf_" + str(round(m_f, 4)) +
            "_ccpa_" + str(round(ccp_a, 6)) +
            "_mse_" + str(round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)) +
            "_r2_" + str(round(sm.r2_score(Y_testing, Y_pred_all), 3))
        )

        # Define the filename with the parameter information
        model_filename = path1 + "/" + model_name + ".joblib"

        # Save the best estimator
        joblib.dump(rf_model, model_filename)

        MAE=round(sm.mean_absolute_error(Y_testing, Y_pred_all), 3)
        MSE=round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)
        RMSE=round(sm.root_mean_squared_error(Y_testing, Y_pred_all), 3)
        R2=round(sm.r2_score(Y_testing, Y_pred_all), 3)
        RES=Y_testing - Y_pred_all
        RES_P=(Y_testing - Y_pred_all) * 100 / Y_testing

        performance_rows.append({
            'Name': name,
            'Fold_out': fold_no_out,
            'Fold_in': fold_no_in,
            # 'Intp_fold': intp_fold,
            'Model_name': model_name,
            'MAE': MAE,
            'MSE': MSE,
            'RMSE': RMSE,
            'R2': R2,
            'RES': RES,
            'RES_P': RES_P
        })

        file_name = "Predictions"+"_kote_"+str(fold_no_out)+"_kova_"+str(fold_no_in)
        my_submission = pd.DataFrame({
            'Actual': Y_testing,
            'Prediction': Y_pred_all,
            'RErr': (Y_testing - Y_pred_all),
            'RErr%': (Y_testing - Y_pred_all) * 100 / Y_testing,
            'Actual(real)': Y_testing * scaler.scale_[inpt_num] + scaler.mean_[inpt_num],
            'Prediction(real)': Y_pred_all * scaler.scale_[inpt_num] + scaler.mean_[inpt_num]
        })
        my_submission.to_csv(os.path.join(path1,file_name+".csv"), index=False)

# save one single summary file for all generated models
performance_summary_df = pd.DataFrame(performance_rows)
performance_summary_df.to_csv(os.path.join(path1, "Perfornace_Summary.csv"), index=False)

# from google.colab import runtime
# runtime.unassign()

# GB

In [ ]:
# GradientBoosting tuning, multiple targets, and pooled denormalized metrics (NoIntp+ManVa)

import os
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import sklearn.metrics as sm
from bayes_opt import BayesianOptimization
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor


# Configuration
names = ["BMP", "COD", "NR", "PR"]

inpt_nums = {
    "BMP": 10,
    "COD": 10,
    "NR": 10,
    "PR": 10,
}

niter = 100     # faster 50. initial 200
initp = 10      # initial 20

k_out = 5
fo_start = 1

parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"


def make_regressor(n_estimators, learning_rate, max_depth,
                   min_samples_split, min_samples_leaf,
                   subsample, max_features):
    return GradientBoostingRegressor(
        loss="squared_error",
        n_estimators=int(n_estimators),
        learning_rate=float(learning_rate),
        max_depth=int(max_depth),
        min_samples_split=int(min_samples_split),
        min_samples_leaf=int(min_samples_leaf),
        subsample=float(subsample),
        max_features=float(max_features),
        random_state=42,
    )


def relative_error_percent(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    error = actual - np.asarray(predicted, dtype=float)

    return np.divide(
        error * 100.0,
        actual,
        out=np.full_like(error, np.nan),
        where=actual != 0,
    )


def metric_values(actual, predicted):
    mse = sm.mean_squared_error(actual, predicted)

    return {
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(mse, 3),
        "RMSE": round(float(np.sqrt(mse)), 3),
        "R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
    }


def pooled_metrics_row(group, scope, fold_out="All"):
    actual = group["Actual(real)"].to_numpy(dtype=float)
    predicted = group["Prediction(real)"].to_numpy(dtype=float)
    residual = actual - predicted
    relative_error = relative_error_percent(actual, predicted)

    return {
        "Name": group["Name"].iloc[0],
        "Scope": scope,
        "Fold_out": fold_out,
        "N_predictions": len(group),
        "Pooled_R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(sm.mean_squared_error(actual, predicted), 3),
        "RE_mean": round(float(np.mean(residual)), 3),
        "RE%_mean": round(float(np.nanmean(relative_error)), 3),
    }


for name in names:
    print(f"===== Running {name} =====")
    inpt_num = inpt_nums[name]

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    path1 = os.path.join(
        parent_dir,
        f"GB {name} Best Models_{timestamp}",
    )
    os.makedirs(path1, exist_ok=True)

    performance_rows = []
    pooled_rows = []

    for fold_no_out in range(fo_start, k_out + 1):
        # Outer held-out test data.
        data2 = pd.read_csv(os.path.join(
            parent_dir,
            f"Testing_data_ori_{name}-fold{fold_no_out}.csv",
        ))

        # Remaining original folds form the outer-training data.
        data = pd.read_csv(parent_dir + 'Training_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')
        # extra_frames = []
        # for fold_num in range(1, k_out + 1):
        #     if fold_num == fold_no_out:
        #         continue
        #     extra_frames.append(pd.read_csv(os.path.join(
        #         parent_dir,
        #         f"Testing_data_ori_{name}-fold{fold_num}.csv",
        #     )))
        # data = pd.concat(extra_frames, axis=0, ignore_index=True)

        # Random 20% of outer-training data as validation.
        data, data3 = train_test_split(
            data,
            test_size=0.20,
            random_state=42,
            shuffle=True,
        )

        # Fit only on training data; transform all three datasets.
        scaler = StandardScaler()
        scaler.fit(data)

        dataset1 = scaler.transform(data)
        dataset2 = scaler.transform(data2)
        dataset3 = scaler.transform(data3)

        X_training = dataset1[:, :inpt_num]
        Y_training = dataset1[:, inpt_num]

        X_testing = dataset2[:, :inpt_num]
        Y_testing = dataset2[:, inpt_num]

        X_valid = dataset3[:, :inpt_num]
        Y_valid = dataset3[:, inpt_num]

        # Bayesian optimization of GB hyperparameters (validation R2).
        def gb_objective(n_estimators, learning_rate, max_depth,
                         min_samples_split, min_samples_leaf,
                         subsample, max_features):
            model = make_regressor(
                n_estimators, learning_rate, max_depth,
                min_samples_split, min_samples_leaf,
                subsample, max_features,
            )
            model.fit(X_training, Y_training)
            prediction = model.predict(X_valid)
            return sm.r2_score(Y_valid, prediction)

        optimizer = BayesianOptimization(
            f=gb_objective,
            pbounds={
                "n_estimators": (100, 1500),
                "learning_rate": (1e-3, 0.5),
                "max_depth": (1, 10),
                "min_samples_split": (2, 20),
                "min_samples_leaf": (1, 10),
                "subsample": (0.5, 1.0),
                "max_features": (0.3, 1.0),
            },
            random_state=1,
        )
        optimizer.maximize(
            n_iter=niter,
            init_points=initp,
        )
        best_params = optimizer.max["params"]

        model = make_regressor(**best_params)
        model.fit(X_training, Y_training)
        prediction = model.predict(X_testing)

        model_name = (
            f"GB_model_{name}"
            f"_kote_{fold_no_out}"
            f"_ne_{int(best_params['n_estimators'])}"
            f"_lr_{float(best_params['learning_rate']):.5f}"
            f"_md_{int(best_params['max_depth'])}"
            f"_mss_{int(best_params['min_samples_split'])}"
            f"_msl_{int(best_params['min_samples_leaf'])}"
            f"_ss_{float(best_params['subsample']):.3f}"
            f"_mf_{float(best_params['max_features']):.3f}"
        )

        joblib.dump(
            model,
            os.path.join(path1, model_name + ".joblib"),
        )

        # Convert target values and predictions back to original units.
        ymean = scaler.mean_[inpt_num]
        yscale = scaler.scale_[inpt_num]

        actual_real = Y_testing * yscale + ymean
        predicted_real = prediction * yscale + ymean
        residual_real = actual_real - predicted_real
        residual_percent_real = relative_error_percent(
            actual_real,
            predicted_real,
        )

        normalized_metrics = metric_values(Y_testing, prediction)
        real_metrics = metric_values(actual_real, predicted_real)

        performance_rows.append({
            "Name": name,
            "Fold_out": fold_no_out,
            "Model_name": model_name,
            "n_estimators": int(best_params["n_estimators"]),
            "learning_rate": round(float(best_params["learning_rate"]), 5),
            "max_depth": int(best_params["max_depth"]),
            "min_samples_split": int(best_params["min_samples_split"]),
            "min_samples_leaf": int(best_params["min_samples_leaf"]),
            "subsample": round(float(best_params["subsample"]), 3),
            "max_features": round(float(best_params["max_features"]), 3),
            "Val_R2": round(optimizer.max["target"], 3),
            **normalized_metrics,
            "MAE_real": real_metrics["MAE"],
            "MSE_real": real_metrics["MSE"],
            "RMSE_real": real_metrics["RMSE"],
            "R2_real": real_metrics["R2"],
            "RE_mean_real": round(float(np.mean(residual_real)), 3),
            "RE%_mean_real": round(float(np.nanmean(residual_percent_real)), 3),
        })

        predictions = pd.DataFrame({
            "Name": name,
            "Fold_out": fold_no_out,
            "Model_name": model_name,
            "Test_row": np.arange(len(Y_testing)),
            "Actual": Y_testing,
            "Prediction": prediction,
            "Actual(real)": actual_real,
            "Prediction(real)": predicted_real,
            "RErr(real)": residual_real,
            "RErr%(real)": residual_percent_real,
        })
        pooled_rows.append(predictions)

        predictions.to_csv(
            os.path.join(path1, f"Predictions_{name}_kote_{fold_no_out}.csv"),
            index=False,
        )

    pd.DataFrame(performance_rows).to_csv(
        os.path.join(path1, "Performance_Summary.csv"),
        index=False,
    )

    pooled_df = pd.concat(pooled_rows, ignore_index=True)
    pooled_df.to_csv(
        os.path.join(path1, "All_Denormalized_Predictions_GB.csv"),
        index=False,
    )

    pooled_metrics = [pooled_metrics_row(pooled_df, "All")]
    for fold_out, group in pooled_df.groupby("Fold_out"):
        pooled_metrics.append(
            pooled_metrics_row(group, "Fold_out", fold_out=fold_out)
        )

    pd.DataFrame(pooled_metrics).to_csv(
        os.path.join(path1, "Pooled_Denormalized_Metrics_GB.csv"),
        index=False,
    )

In [ ]:
# 1 (small)- GradientBoosting hyperparameter tuninig using BayesianOptimization in 2 steps
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor

import os
import numpy as np
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import sklearn.metrics as sm
import pandas as pd

from bayes_opt import BayesianOptimization
import joblib

import datetime, os
from datetime import datetime

# Clear any logs from previous runs
# !rm -rf ./logs/
# %load_ext tensorboard

#modeling parameters
model_num=1 #10 #how many model want to save
# iter_num=20 #35 #max iteration number
# tr_fld_num=1 # Training data K-fold number
# epchs=150 #500 #800

niter=200 # how many iter bo will run
initp=20  # how many initial iteration

# stp=0 #0=BM, 1=COD, 2=TN, 3=TP
inpt_num=10 #how mamy input the model has

# names = ["Prd", "OR", "NR", "PR"]
name="PR" #"Prd", "OR", "NR", "PR"
k_out   = 5   # outer folds
k_out1   = 5   # k_out, adjusted outer folds
k_out2   = 5   # k_out, adjusted outer folds
k_in    = 5   # 5, inner folds
fo_start=5 #1,starting fold number for outer loop
fi_start=5 #starting fold number for inner loop
k_in_start=5

stp=3 #0=Prd, 1=OR, 2=NR, 3=PR

# actf='relu' #'relu' 'tanh' 'sigmoid'
# nmin=16
# nmax=128 #256#128
# nstp=8
# lr=1e-2 #1e-2 1e-3 1e-1

#ploting parameters
# unt="" #" (mg/L)"
# data_points=20 #total testing data point (rounded)
# err_lim1=25 #residula % error limit shown in the plot
# err_lim=0.25 #residula error limit shown in the plot
# xmax=3

# Parent Directory path
# parent_dir = "/content/drive/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
# dir_name='my_dir1' # directory name wherer the models will be saved

# Safe timestamp for Windows folder names
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
new_folder = f"GBoost {name} Best Models_{timestamp}"
# new_folder="Actf="+str(actf)+" Lr="+str(lr)+" "+name+" Best Models" #" 1st Architecture" #" Best Modeles"

# make one shared output folder for all models
path1 = os.path.join(parent_dir, new_folder)
os.makedirs(path1, exist_ok=True)


#output statistical info
norm_prmt = pd.read_csv(parent_dir+'Normalizing_parameters.csv').values
mean9=norm_prmt[inpt_num,stp] #row,col
stdev9=norm_prmt[2*inpt_num+1,stp]

# collect all performance rows here, then save one CSV at the end
performance_rows = []

for fold_no_out in range(fo_start, k_out1 + 1):
    # loading original testing data
    # data2 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')
    # data2 = pd.read_csv(parent_dir + 'K=5 folds '+name+'/Training_data_'+name+'-fold'+str(fold_no_out)+'.csv')
    data2 = pd.read_csv(parent_dir+'Normalized_Data-AMBR_ori_v3_pp_num_'+name+'.csv')

    for fold_no_in in range(fi_start, k_out2 + 1):
        # if fold_no_out == fold_no_in:
        #     continue

        # loading original validation data
        # data3 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_in) + '.csv')
        data3 = pd.read_csv(parent_dir+'Normalized_Data-AMBR_ori_v3_pp_num_'+name+'.csv')

        # # loading original training data from the remaining folds
        # extra_frames = []
        # for num in range(1, k_out + 1):
        #     if num == fold_no_out or num == fold_no_in:
        #         continue
        #     extra_frames.append(
        #         pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(num) + '.csv')
        #     )

        # if len(extra_frames) > 0:
        #     data_extra_all = pd.concat(extra_frames, axis=0, ignore_index=True)
        # else:
        #     data_extra_all = None

        for intp_fold in range(k_in_start, k_in + 1):

            # output statistical info
            # norm_prmt = pd.read_csv(parent_dir+'Normalizing_parameters.csv').values
            # mean9 = norm_prmt[inpt_num, stp]
            # stdev9 = norm_prmt[2*inpt_num+1, stp]

            # loading traninig interpolated data
            # data = pd.read_csv(
            #     parent_dir + 'Ko=' + str(fold_no_out) + ' Ki=' + str(intp_fold) + ' ' + name +
            #     ' intp/Training_data_int_'+name+'-foo' + str(fold_no_out) + '-foi' + str(intp_fold) + '.csv'
            # )
            data = pd.read_csv(parent_dir+'K=5 folds '+name+'/Training_data_'+name+'-fold'+str(intp_fold)+'.csv')

            # # merging the remaining original testing folds outside the loop with training data
            # if data_extra_all is not None:
            #     # Stack one after another (row-wise / vertical append)
            #     data = pd.concat([data,data_extra_all], axis=0, ignore_index=True)
            #     # Optional: remove exact duplicate rows after merge
            #     data = data.drop_duplicates().reset_index(drop=True)
            
            # data_nontesting =pd.concat([data, data3], axis=0, ignore_index=True)
            # scaler = StandardScaler()
            # scaler.fit(data_nontesting)

            # data_norm = scaler.transform(data) #training data normalization
            # dataset1 = data_norm
            dataset1 = data.values
            X_training = dataset1[:, 0:inpt_num]
            Y_training = dataset1[:, inpt_num]

            # data2_norm = scaler.transform(data2) #testing data normalization
            # dataset2 = data2_norm
            dataset2 = data2.values
            X_testing = dataset2[:, 0:inpt_num]
            Y_testing = dataset2[:, inpt_num]

            # data3_norm = scaler.transform(data3) #validation data normalization
            # dataset3 = data3_norm
            dataset3 = data3.values
            X_valid = dataset3[:, 0:inpt_num]
            Y_valid = dataset3[:, inpt_num]
            

            # 1) Stage-1 BO: n_estimators, learning_rate, max_depth
            def bo_params_gbr_stage1(n_estimators, learning_rate, max_depth):
                params = {
                    'n_estimators': int(round(n_estimators)),
                    'learning_rate': learning_rate,
                    'max_depth': int(round(max_depth)),
                    'random_state': 1
                }

                gbr_model = GradientBoostingRegressor(**params)
                gbr_model.fit(X_training, Y_training)
                Y_pred_valid = gbr_model.predict(X_valid)

                scores = sm.r2_score(Y_valid, Y_pred_valid)
                return scores

            gbr_bo = BayesianOptimization(
                bo_params_gbr_stage1,
                {
                    'n_estimators': (100, 1500),
                    'learning_rate': (1e-3, 0.3),
                    'max_depth': (2, 10)
                },
                random_state=1
            )

            results = gbr_bo.maximize(n_iter=niter, init_points=initp)

            n_est = int(round(gbr_bo.max['params']['n_estimators']))
            l_r = gbr_bo.max['params']['learning_rate']
            m_d = int(round(gbr_bo.max['params']['max_depth']))

            # 2) Stage-2 BO: min_samples_split, min_samples_leaf, subsample, max_features
            def bo_params_gbr_stage2(min_samples_split, min_samples_leaf, subsample, max_features):
                params = {
                    'n_estimators': n_est,
                    'learning_rate': l_r,
                    'max_depth': m_d,
                    'min_samples_split': int(round(min_samples_split)),
                    'min_samples_leaf': int(round(min_samples_leaf)),
                    'subsample': subsample,
                    'max_features': max_features,
                    'random_state': 1
                }

                gbr_model = GradientBoostingRegressor(**params)
                gbr_model.fit(X_training, Y_training)
                Y_pred_valid = gbr_model.predict(X_valid)

                scores = sm.r2_score(Y_valid, Y_pred_valid)
                return scores

            gbr_bo = BayesianOptimization(
                bo_params_gbr_stage2,
                {
                    'min_samples_split': (2, 20),
                    'min_samples_leaf': (1, 10),
                    'subsample': (0.5, 1.0),
                    'max_features': (0.3, 1.0)
                },
                random_state=1
            )

            results = gbr_bo.maximize(n_iter=niter, init_points=initp)

            best_params_dict = gbr_bo.max['params']
            m_s_s = int(round(best_params_dict['min_samples_split']))
            m_s_l = int(round(best_params_dict['min_samples_leaf']))
            s_s = best_params_dict['subsample']
            m_f = best_params_dict['max_features']

            # Final GBR model with tuned parameters
            gbr_model = GradientBoostingRegressor(
                n_estimators=n_est,
                learning_rate=l_r,
                max_depth=m_d,
                min_samples_split=m_s_s,
                min_samples_leaf=m_s_l,
                subsample=s_s,
                max_features=m_f,
                random_state=1
            )

            # Train the model
            gbr_model.fit(X_training, Y_training)

            # Make predictions on test data
            Y_pred_all = gbr_model.predict(X_testing)

            model_name = (
                "GBR_model_" + name +
                "_kote_" + str(fold_no_out) +
                "_kova_" + str(fold_no_in) +
                "_kitr_" + str(intp_fold) +
                "_estm_" + str(n_est) +
                "_lr_" + str(round(l_r, 6)) +
                "_mxdep_" + str(m_d) +
                "_mss_" + str(m_s_s) +
                "_msl_" + str(m_s_l) +
                "_subs_" + str(round(s_s, 4)) +
                "_mxf_" + str(round(m_f, 4)) +
                "_mse_" + str(round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)) +
                "_r2_" + str(round(sm.r2_score(Y_testing, Y_pred_all), 3))
            )

            # Define the filename with the parameter information
            model_filename = path1 + "/" + model_name + ".joblib"

            # Save the best estimator
            joblib.dump(gbr_model, model_filename)

            MAE=round(sm.mean_absolute_error(Y_testing, Y_pred_all), 3)
            MSE=round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)
            RMSE=round(sm.root_mean_squared_error(Y_testing, Y_pred_all), 3)
            R2=round(sm.r2_score(Y_testing, Y_pred_all), 3)
            RES=Y_testing - Y_pred_all
            RES_P=(Y_testing - Y_pred_all) * 100 / Y_testing

            performance_rows.append({
                'Name': name,
                'Fold_out': fold_no_out,
                'Fold_in': fold_no_in,
                'Intp_fold': intp_fold,
                'Model_name': model_name,
                'MAE': MAE,
                'MSE': MSE,
                'RMSE': RMSE,
                'R2': R2,
                'RES': RES,
                'RES_P': RES_P
            })

            file_name = "Predictions"+"_"+name+"_GBoost"+"_kote_"+str(fold_no_out)+"_kova_"+str(fold_no_in)+"_kitr_"+str(intp_fold)
            my_submission = pd.DataFrame({
                'Actual': Y_testing,
                'Prediction': Y_pred_all,
                'RErr': (Y_testing - Y_pred_all),
                'RErr%': (Y_testing - Y_pred_all) * 100 / Y_testing,
                # 'Actual(real)': Y_testing * scaler.scale_[13] + scaler.mean_[13],
                # 'Prediction(real)': Y_pred_all * scaler.scale_[13] + scaler.mean_[13]
                'Actual(real)': Y_testing*stdev9+mean9,
                'Prediction(real)': Y_pred_all*stdev9+mean9
            })
            my_submission.to_csv(os.path.join(path1,file_name+".csv"), index=False)

# save one single summary file for all generated models
performance_summary_df = pd.DataFrame(performance_rows)
performance_summary_df.to_csv(os.path.join(path1, "Perfornace_Summary.csv"), index=False)

# from google.colab import runtime
# runtime.unassign()

## No CV for interpolation training

In [ ]:
# 1 (small)- GradientBoosting hyperparameter tuninig using BayesianOptimization in 2 steps
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor

import os
import numpy as np
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import sklearn.metrics as sm
import pandas as pd

from bayes_opt import BayesianOptimization
import joblib

import datetime, os
from datetime import datetime

# Clear any logs from previous runs
# !rm -rf ./logs/
# %load_ext tensorboard

#modeling parameters
model_num=1 #10 #how many model want to save
# iter_num=20 #35 #max iteration number
# tr_fld_num=1 # Training data K-fold number
# epchs=150 #500 #800

niter=200 # how many iter bo will run
initp=20  # how many initial iteration

# stp=0 #0=BM, 1=COD, 2=TN, 3=TP
inpt_num=13 #how mamy input the model has

# names = ["BM", "TP", "TN", "COD"]
name="TP" #"BM", "TP", "TN", "COD"
k_out   = 10   # outer folds
k_out1   = 10   # k_out, adjusted outer folds
k_out2   = 10   # k_out, adjusted outer folds
fo_start=10 #1,starting fold number for outer loop
fi_start=1 #starting fold number for inner loop

Inst="NoIntpCV"
#ploting parameters
# unt="" #" (mg/L)"
# data_points=20 #total testing data point (rounded)
# err_lim1=25 #residula % error limit shown in the plot
# err_lim=0.25 #residula error limit shown in the plot
# xmax=3

# Parent Directory path
# parent_dir = "/content/drive/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"#+new_folder+"/"
# dir_name='my_dir1' # directory name wherer the models will be saved

# Safe timestamp for Windows folder names
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
new_folder = f"GBR {name} Best Models_{timestamp} {Inst}"

# make one shared output folder for all models
path1 = os.path.join(parent_dir, new_folder)
os.makedirs(path1, exist_ok=True)

# collect all performance rows here, then save one CSV at the end
performance_rows = []

for fold_no_out in range(fo_start, k_out1 + 1):
    # loading original testing data
    data2 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')

    for fold_no_in in range(fi_start, k_out2 + 1):
        if fold_no_out == fold_no_in:
            continue

        # loading original validation data
        data3 = pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(fold_no_in) + '.csv')

        # loading original training data from the remaining folds
        extra_frames = []
        for num in range(1, k_out + 1):
            if num == fold_no_out or num == fold_no_in:
                continue
            extra_frames.append(
                pd.read_csv(parent_dir + 'Testing_data_ori_' + name + '-fold' + str(num) + '.csv')
            )

        if len(extra_frames) > 0:
            data_extra_all = pd.concat(extra_frames, axis=0, ignore_index=True)
        else:
            data_extra_all = None


        # loading traninig interpolated data
        data = pd.read_csv(
        parent_dir
        + "Inerpolated_Data_"
        + name
        + "_fold"
        + str(fold_no_out)
        + "_fold"
        + str(fold_no_in)
        + ".csv"
        )

        # merging the remaining original testing folds outside the loop with training data
        if data_extra_all is not None:
            # Stack one after another (row-wise / vertical append)
            data = pd.concat([data,data_extra_all], axis=0, ignore_index=True)
            # Optional: remove exact duplicate rows after merge
            data = data.drop_duplicates().reset_index(drop=True)

        # data_nontesting =pd.concat([data, data3], axis=0, ignore_index=True)
        scaler = StandardScaler()
        scaler.fit(data)

        data_norm = scaler.transform(data) #training data normalization
        dataset1 = data_norm
        X_training = dataset1[:, 0:inpt_num]
        Y_training = dataset1[:, inpt_num]

        data2_norm = scaler.transform(data2) #testing data normalization
        dataset2 = data2_norm
        X_testing = dataset2[:, 0:inpt_num]
        Y_testing = dataset2[:, inpt_num]

        data3_norm = scaler.transform(data3) #validation data normalization
        dataset3 = data3_norm
        X_valid = dataset3[:, 0:inpt_num]
        Y_valid = dataset3[:, inpt_num]

        # 1) Stage-1 BO: n_estimators, learning_rate, max_depth
        def bo_params_gbr_stage1(n_estimators, learning_rate, max_depth):
            params = {
                'n_estimators': int(round(n_estimators)),
                'learning_rate': learning_rate,
                'max_depth': int(round(max_depth)),
                'random_state': 1
            }

            gbr_model = GradientBoostingRegressor(**params)
            gbr_model.fit(X_training, Y_training)
            Y_pred_valid = gbr_model.predict(X_valid)

            scores = sm.r2_score(Y_valid, Y_pred_valid)
            return scores

        gbr_bo = BayesianOptimization(
            bo_params_gbr_stage1,
            {
                'n_estimators': (100, 1500),
                'learning_rate': (1e-3, 0.3),
                'max_depth': (2, 10)
            },
            random_state=1
        )

        results = gbr_bo.maximize(n_iter=niter, init_points=initp)

        n_est = int(round(gbr_bo.max['params']['n_estimators']))
        l_r = gbr_bo.max['params']['learning_rate']
        m_d = int(round(gbr_bo.max['params']['max_depth']))

        # 2) Stage-2 BO: min_samples_split, min_samples_leaf, subsample, max_features
        def bo_params_gbr_stage2(min_samples_split, min_samples_leaf, subsample, max_features):
            params = {
                'n_estimators': n_est,
                'learning_rate': l_r,
                'max_depth': m_d,
                'min_samples_split': int(round(min_samples_split)),
                'min_samples_leaf': int(round(min_samples_leaf)),
                'subsample': subsample,
                'max_features': max_features,
                'random_state': 1
            }

            gbr_model = GradientBoostingRegressor(**params)
            gbr_model.fit(X_training, Y_training)
            Y_pred_valid = gbr_model.predict(X_valid)

            scores = sm.r2_score(Y_valid, Y_pred_valid)
            return scores

        gbr_bo = BayesianOptimization(
            bo_params_gbr_stage2,
            {
                'min_samples_split': (2, 20),
                'min_samples_leaf': (1, 10),
                'subsample': (0.5, 1.0),
                'max_features': (0.3, 1.0)
            },
            random_state=1
        )

        results = gbr_bo.maximize(n_iter=niter, init_points=initp)

        best_params_dict = gbr_bo.max['params']
        m_s_s = int(round(best_params_dict['min_samples_split']))
        m_s_l = int(round(best_params_dict['min_samples_leaf']))
        s_s = best_params_dict['subsample']
        m_f = best_params_dict['max_features']

        # Final GBR model with tuned parameters
        gbr_model = GradientBoostingRegressor(
            n_estimators=n_est,
            learning_rate=l_r,
            max_depth=m_d,
            min_samples_split=m_s_s,
            min_samples_leaf=m_s_l,
            subsample=s_s,
            max_features=m_f,
            random_state=1
        )

        # Train the model
        gbr_model.fit(X_training, Y_training)

        # Make predictions on test data
        Y_pred_all = gbr_model.predict(X_testing)

        model_name = (
            "GBR_model_" + name +
            "_kote_" + str(fold_no_out) +
            "_kova_" + str(fold_no_in) +
            # "_kitr_" + str(intp_fold) +
            "_estm_" + str(n_est) +
            "_lr_" + str(round(l_r, 6)) +
            "_mxdep_" + str(m_d) +
            "_mss_" + str(m_s_s) +
            "_msl_" + str(m_s_l) +
            "_subs_" + str(round(s_s, 4)) +
            "_mxf_" + str(round(m_f, 4)) +
            "_mse_" + str(round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)) +
            "_r2_" + str(round(sm.r2_score(Y_testing, Y_pred_all), 3))
        )

        # Define the filename with the parameter information
        model_filename = path1 + "/" + model_name + ".joblib"

        # Save the best estimator
        joblib.dump(gbr_model, model_filename)

        MAE=round(sm.mean_absolute_error(Y_testing, Y_pred_all), 3)
        MSE=round(sm.mean_squared_error(Y_testing, Y_pred_all), 3)
        RMSE=round(sm.root_mean_squared_error(Y_testing, Y_pred_all), 3)
        R2=round(sm.r2_score(Y_testing, Y_pred_all), 3)
        RES=Y_testing - Y_pred_all
        RES_P=(Y_testing - Y_pred_all) * 100 / Y_testing

        performance_rows.append({
            'Name': name,
            'Fold_out': fold_no_out,
            'Fold_in': fold_no_in,
            # 'Intp_fold': intp_fold,
            'Model_name': model_name,
            'MAE': MAE,
            'MSE': MSE,
            'RMSE': RMSE,
            'R2': R2,
            'RES': RES,
            'RES_P': RES_P
        })

        file_name = "Predictions"+"_kote_"+str(fold_no_out)+"_kova_"+str(fold_no_in)
        my_submission = pd.DataFrame({
            'Actual': Y_testing,
            'Prediction': Y_pred_all,
            'RErr': (Y_testing - Y_pred_all),
            'RErr%': (Y_testing - Y_pred_all) * 100 / Y_testing,
            'Actual(real)': Y_testing * scaler.scale_[inpt_num] + scaler.mean_[inpt_num],
            'Prediction(real)': Y_pred_all * scaler.scale_[inpt_num] + scaler.mean_[inpt_num]
        })
        my_submission.to_csv(os.path.join(path1,file_name+".csv"), index=False)

# save one single summary file for all generated models
performance_summary_df = pd.DataFrame(performance_rows)
performance_summary_df.to_csv(os.path.join(path1, "Perfornace_Summary.csv"), index=False)

# from google.colab import runtime
# runtime.unassign()

# Ridge

In [ ]:
# Ridge tuning, multiple targets, and pooled denormalized metrics (NoIntp+ManVa)

import os
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import sklearn.metrics as sm
from bayes_opt import BayesianOptimization
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge


# Configuration
names = ["BMP", "COD", "NR", "PR"]

inpt_nums = {
    "BMP": 10,
    "COD": 10,
    "NR": 10,
    "PR": 10,
}

niter = 200      # Ridge is very fast; raise freely
initp = 20

k_out = 5
fo_start = 1

parent_dir = "E:/My Drive/Colab Notebooks/AMBR Prediction_v5kcv/"


def make_regressor(log_alpha):
    # alpha is searched on a log10 scale
    return Ridge(alpha=10 ** log_alpha, random_state=42)


def relative_error_percent(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    error = actual - np.asarray(predicted, dtype=float)

    return np.divide(
        error * 100.0,
        actual,
        out=np.full_like(error, np.nan),
        where=actual != 0,
    )


def metric_values(actual, predicted):
    mse = sm.mean_squared_error(actual, predicted)

    return {
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(mse, 3),
        "RMSE": round(float(np.sqrt(mse)), 3),
        "R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
    }


def pooled_metrics_row(group, scope, fold_out="All"):
    actual = group["Actual(real)"].to_numpy(dtype=float)
    predicted = group["Prediction(real)"].to_numpy(dtype=float)
    residual = actual - predicted
    relative_error = relative_error_percent(actual, predicted)

    return {
        "Name": group["Name"].iloc[0],
        "Scope": scope,
        "Fold_out": fold_out,
        "N_predictions": len(group),
        "Pooled_R2": (
            round(sm.r2_score(actual, predicted), 3)
            if len(actual) > 1 else np.nan
        ),
        "MAE": round(sm.mean_absolute_error(actual, predicted), 3),
        "MSE": round(sm.mean_squared_error(actual, predicted), 3),
        "RE_mean": round(float(np.mean(residual)), 3),
        "RE%_mean": round(float(np.nanmean(relative_error)), 3),
    }


for name in names:
    print(f"===== Running {name} =====")
    inpt_num = inpt_nums[name]

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    path1 = os.path.join(
        parent_dir,
        f"Ridge {name} Best Models_{timestamp}",
    )
    os.makedirs(path1, exist_ok=True)

    performance_rows = []
    pooled_rows = []

    for fold_no_out in range(fo_start, k_out + 1):
        # Outer held-out test data.
        data2 = pd.read_csv(os.path.join(
            parent_dir,
            f"Testing_data_ori_{name}-fold{fold_no_out}.csv",
        ))

        # Remaining original folds form the outer-training data.
        data = pd.read_csv(parent_dir + 'Training_data_ori_' + name + '-fold' + str(fold_no_out) + '.csv')
        # extra_frames = []
        # for fold_num in range(1, k_out + 1):
        #     if fold_num == fold_no_out:
        #         continue
        #     extra_frames.append(pd.read_csv(os.path.join(
        #         parent_dir,
        #         f"Testing_data_ori_{name}-fold{fold_num}.csv",
        #     )))
        # data = pd.concat(extra_frames, axis=0, ignore_index=True)

        # Random 20% of outer-training data as validation.
        data, data3 = train_test_split(
            data,
            test_size=0.20,
            random_state=42,
            shuffle=True,
        )

        # Fit only on training data; transform all three datasets.
        scaler = StandardScaler()
        scaler.fit(data)

        dataset1 = scaler.transform(data)
        dataset2 = scaler.transform(data2)
        dataset3 = scaler.transform(data3)

        X_training = dataset1[:, :inpt_num]
        Y_training = dataset1[:, inpt_num]

        X_testing = dataset2[:, :inpt_num]
        Y_testing = dataset2[:, inpt_num]

        X_valid = dataset3[:, :inpt_num]
        Y_valid = dataset3[:, inpt_num]

        # Bayesian optimization of alpha (validation R2).
        def ridge_objective(log_alpha):
            model = make_regressor(log_alpha)
            model.fit(X_training, Y_training)
            prediction = model.predict(X_valid)
            return sm.r2_score(Y_valid, prediction)

        optimizer = BayesianOptimization(
            f=ridge_objective,
            pbounds={
                "log_alpha": (-4, 4),     # alpha: 1e-4 - 1e4
            },
            random_state=1,
        )
        optimizer.maximize(
            n_iter=niter,
            init_points=initp,
        )
        best_params = optimizer.max["params"]

        model = make_regressor(**best_params)
        model.fit(X_training, Y_training)
        prediction = model.predict(X_testing)

        model_name = (
            f"Ridge_model_{name}"
            f"_kote_{fold_no_out}"
            f"_alpha_{10 ** best_params['log_alpha']:.4g}"
        )

        joblib.dump(
            model,
            os.path.join(path1, model_name + ".joblib"),
        )

        # Convert target values and predictions back to original units.
        ymean = scaler.mean_[inpt_num]
        yscale = scaler.scale_[inpt_num]

        actual_real = Y_testing * yscale + ymean
        predicted_real = prediction * yscale + ymean
        residual_real = actual_real - predicted_real
        residual_percent_real = relative_error_percent(
            actual_real,
            predicted_real,
        )

        normalized_metrics = metric_values(Y_testing, prediction)
        real_metrics = metric_values(actual_real, predicted_real)

        performance_rows.append({
            "Name": name,
            "Fold_out": fold_no_out,
            "Model_name": model_name,
            "alpha": round(10 ** best_params["log_alpha"], 5),
            "Val_R2": round(optimizer.max["target"], 3),
            **normalized_metrics,
            "MAE_real": real_metrics["MAE"],
            "MSE_real": real_metrics["MSE"],
            "RMSE_real": real_metrics["RMSE"],
            "R2_real": real_metrics["R2"],
            "RE_mean_real": round(float(np.mean(residual_real)), 3),
            "RE%_mean_real": round(float(np.nanmean(residual_percent_real)), 3),
        })

        predictions = pd.DataFrame({
            "Name": name,
            "Fold_out": fold_no_out,
            "Model_name": model_name,
            "Test_row": np.arange(len(Y_testing)),
            "Actual": Y_testing,
            "Prediction": prediction,
            "Actual(real)": actual_real,
            "Prediction(real)": predicted_real,
            "RErr(real)": residual_real,
            "RErr%(real)": residual_percent_real,
        })
        pooled_rows.append(predictions)

        predictions.to_csv(
            os.path.join(path1, f"Predictions_{name}_kote_{fold_no_out}.csv"),
            index=False,
        )

    pd.DataFrame(performance_rows).to_csv(
        os.path.join(path1, "Performance_Summary.csv"),
        index=False,
    )

    pooled_df = pd.concat(pooled_rows, ignore_index=True)
    pooled_df.to_csv(
        os.path.join(path1, "All_Denormalized_Predictions_Ridge.csv"),
        index=False,
    )

    pooled_metrics = [pooled_metrics_row(pooled_df, "All")]
    for fold_out, group in pooled_df.groupby("Fold_out"):
        pooled_metrics.append(
            pooled_metrics_row(group, "Fold_out", fold_out=fold_out)
        )

    pd.DataFrame(pooled_metrics).to_csv(
        os.path.join(path1, "Pooled_Denormalized_Metrics_Ridge.csv"),
        index=False,
    )